In [ ]:
# ── CAPTURE CELL : Run this FIRST before any other cell ─────────────────────
# All print() output appears in BOTH the cell output AND ps2_outputs/console.log
# All plt.show() figures appear in BOTH the cell output AND ps2_outputs/<name>.png
import os, sys
OUT_DIR = "ps2_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

class _Tee:
    """Writes to Jupyter cell output + log file simultaneously."""
    def __init__(self, jupyter_stream, log_fh):
        self._jupyter = jupyter_stream   # Jupyter OutStream -> shows in cell
        self._log     = log_fh           # file handle       -> saved to disk
        self._is_tee  = True
    def write(self, data):
        try:
            self._jupyter.write(data)
            self._jupyter.flush()
        except Exception:
            pass
        try:
            self._log.write(data)
            self._log.flush()
        except Exception:
            pass
    def flush(self):
        try: self._jupyter.flush()
        except Exception: pass
        try: self._log.flush()
        except Exception: pass
    def __getattr__(self, name):
        return getattr(self._jupyter, name)

if not getattr(sys.stdout, "_is_tee", False):
    _log_fh    = open(os.path.join(OUT_DIR, "console.log"), "w", buffering=1, encoding="utf-8")
    sys.stdout = _Tee(sys.stdout, _log_fh)
    sys.stderr = _Tee(sys.stderr, _log_fh)
    print("[capture] All output -> cell output + " + os.path.join(OUT_DIR, "console.log"))
else:
    print("[capture] Already active.")

try:
    import matplotlib.pyplot as _plt
    if not getattr(_plt.show, "_wrapped", False):
        _orig_show = _plt.show
        # Counter stored as mutable default arg on the function itself --
        # immune to any variable-name collision from other cells in the notebook.
        def _show(*a, _cnt=[0], **k):
            for _fig_num in _plt.get_fignums():
                _fig = _plt.figure(_fig_num)
                _title = (
                    (_fig._suptitle.get_text().strip() if _fig._suptitle else None)
                    or (_fig.axes[0].get_title().strip() if _fig.axes else None)
                    or f"fig_{_cnt[0]:02d}"
                )
                _safe = "".join(
                    c if c.isalnum() or c in "-_ " else "_" for c in _title
                ).strip().replace(" ", "_")[:60]
                _fname = os.path.join(OUT_DIR, f"{_safe}_{_cnt[0]:02d}.png")
                try:
                    _fig.savefig(_fname, dpi=110, bbox_inches="tight")
                    print(f"[capture] fig saved -> {_fname}")
                    _cnt[0] += 1
                except Exception as _save_err:
                    print(f"[capture] fig save failed: {_save_err}")
            return _orig_show(*a, **k)
        _show._wrapped = True
        _plt.show = _show
    print(f"[capture] matplotlib patched -> figures saved to {OUT_DIR}/")
except Exception as _cap_err:
    print("[capture] matplotlib not ready:", _cap_err)


**PS2 — Cascade Chain Pattern Analysis**

---
SageMaker Studio | MLflow | Feature Store

Architecture : CUBIC MARS AWS v6
Gold table   : mars_dev.gold.device_ps2_chains
Grain        : one row per (device_id, transit_day) where chain_length >= 2
               — cascade days only; non-cascade days are NOT in this table

Purpose      : Understand HOW cascades propagate — not predict them.
               Time windows (0-5 / 5-15 / 15-30 / 30-60 / 60+ min)
               reveal propagation speed, subsystem co-failure patterns,
               and severity regime transitions per device category.

Analysis     : Time-window propagation | Phi correlation | Conditional P(B|A,T)
               Markov chain | HMM 3-state | Association rules | Network graph
Device scope : TVM | GATE (RVG/HBG/SAG) | VALIDATOR (BMV bus)
Feasibility  : 85%  (15% gap: SVN_STAGE.CMDB_CI / CMDB_REL = 0 rows)

---

In [ ]:
# ── CELL 1 : Install dependencies ─────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "s3fs", "pyarrow",
    "scikit-learn>=1.4", "xgboost>=2.0", "lightgbm>=4.3",
    "hmmlearn>=0.3", "mlxtend>=0.23",
    "optuna>=3.6", "mlflow>=2.13", "sagemaker-mlflow",
    "boto3", "sagemaker>=2.220,<3.0",
    "pandas>=2.0", "matplotlib", "seaborn", "networkx", "scipy",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
print("OK")

In [ ]:
# ── CELL 2 : Imports ──────────────────────────────────────────────────────────
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import networkx as nx
import boto3, sagemaker, joblib
import scipy.stats as stats

from collections import Counter, defaultdict
from itertools import combinations

from hmmlearn  import hmm
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing     import TransactionEncoder

import mlflow
try:
    from sagemaker import Session
except ImportError:
    from sagemaker.session import Session
try:
    from sagemaker import get_execution_role
except ImportError:
    from sagemaker.session import get_execution_role
from sagemaker.feature_store.feature_group      import FeatureGroup
from sagemaker.feature_store.feature_definition import (FeatureDefinition,
                                                        FeatureTypeEnum)
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
print("Imports OK")

In [ ]:
# ── CELL 3 : Configuration ────────────────────────────────────────────────────
# Analysis-only notebook — no SageMaker endpoint or model registry.
# Outputs: MLflow experiment with artifact charts, CSVs, matrices.

ARTIFACT_BUCKET = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
GOLD_BUCKET     = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX     = "chicago/gold/device_ps2_chains"
GOLD_S3         = f"s3://{GOLD_BUCKET}/{GOLD_PREFIX}"

SM_SESSION = Session()
ROLE       = get_execution_role()
REGION     = boto3.Session().region_name
SM_CLIENT  = boto3.client("sagemaker", region_name=REGION)
S3_CLIENT  = boto3.client("s3",        region_name=REGION)

MLFLOW_SERVER_ARN = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN

try:
    MLFLOW_TRACKING_URI = SM_CLIENT.describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev"
    )["TrackingServerUrl"]
    print(f"MLflow managed server: {MLFLOW_TRACKING_URI}")
except Exception:
    MLFLOW_TRACKING_URI = "file:///home/sagemaker-user/mlruns"
    print("MLflow managed server not found — using local file fallback")
mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)

EXPERIMENT_NAME = "chicago-ps2-cascade-analysis"
FS_GROUP_NAME   = "chicago-ps2-features"

# Time-window buckets for cascade propagation analysis
TIME_WINDOWS = {
    "0-5 min"  : (0,   5),
    "5-15 min" : (5,  15),
    "15-30 min": (15, 30),
    "30-60 min": (30, 60),
    "60+ min"  : (60, 9999),
}

DEVICE_CATEGORIES = ["TVM", "GATE", "VALIDATOR"]

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Gold S3 : {GOLD_S3}")
print(f"Time windows: {list(TIME_WINDOWS.keys())}")

In [ ]:
# ── CELL 3b : Checkpoint — load saved state (skip Cells 4-6 if found) ────────
# Saves: df, train_df, val_df, test_df, FEATURE_COLS, SECONDARY_TARGETS
# Path : /home/sagemaker-user/PS2/checkpoints/  (EFS — survives instance stop)
# Usage: Run Cells 1-3-3b → if CHECKPOINT_LOADED=True → jump to Cell 10
#        Set FORCE_RELOAD=True to re-pull from S3 and rebuild features.

import os, joblib, pandas as _pd_ckpt

CKPT_DIR      = "/home/sagemaker-user/PS2/checkpoints"
FORCE_RELOAD  = False
CHECKPOINT_LOADED = False

_ckpt_df    = os.path.join(CKPT_DIR, "df.parquet")
_ckpt_meta  = os.path.join(CKPT_DIR, "meta.joblib")

if not FORCE_RELOAD and os.path.exists(_ckpt_df) and os.path.exists(_ckpt_meta):
    print("[CHECKPOINT] Loading from", CKPT_DIR)
    df       = _pd_ckpt.read_parquet(_ckpt_df)
    train_df = _pd_ckpt.read_parquet(os.path.join(CKPT_DIR, "train_df.parquet"))
    val_df   = _pd_ckpt.read_parquet(os.path.join(CKPT_DIR, "val_df.parquet"))
    test_df  = _pd_ckpt.read_parquet(os.path.join(CKPT_DIR, "test_df.parquet"))
    _meta    = joblib.load(_ckpt_meta)
    FEATURE_COLS       = _meta["FEATURE_COLS"]
    SECONDARY_TARGETS  = _meta["SECONDARY_TARGETS"]
    EXCLUDE_COLS       = _meta["EXCLUDE_COLS"]
    CHECKPOINT_LOADED  = True
    print(f"  df           : {df.shape}")
    print(f"  train / val / test : {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")
    print(f"  FEATURE_COLS : {len(FEATURE_COLS)} features")
    print()
    print("  CHECKPOINT LOADED — skip Cells 4, 5, 6 and jump to Cell 7 (Feature Store)")
    print("  or Cell 10 (Cascade Propagation) if Feature Store is not needed.")
else:
    print("[CHECKPOINT] Not found or FORCE_RELOAD=True — running full pipeline (Cells 4-6).")
    print("  Checkpoint will be saved automatically at end of Cell 6.")


In [ ]:
# ── CELL 4 : Load Gold table (Parquet on S3) ──────────────────────────────────
print(f"Loading Gold table from: {GOLD_S3}")

df = pd.read_parquet(GOLD_S3, storage_options={"client_kwargs": {"region_name": REGION}})
GOLD_VERSION = None   # Parquet has no version; use None for MLflow tags
DATA_SOURCE  = "parquet"

df["transit_day"] = pd.to_datetime(df["transit_day"])

# Delta→Parquet exports sometimes carry numeric cols as object dtype — force cast
_NUMERIC_COLS = [
    "chain_length", "chain_span_min", "critical_in_chain", "oos_in_chain",
    "distinct_subsystems", "has_cash_cascade", "has_gate_mech_in_chain",
    "has_printer_in_chain", "has_csc_reader_in_chain", "has_dopp_in_chain",
    "has_scrst_in_chain", "has_cascade",
]
for _c in _NUMERIC_COLS:
    if _c in df.columns and not pd.api.types.is_numeric_dtype(df[_c]):
        df[_c] = pd.to_numeric(df[_c], errors="coerce")
        print(f"  cast {_c} → numeric")

print(f"Parquet source  : confirmed")
print(f"Shape           : {df.shape}")
print(f"Date range      : {df['transit_day'].min().date()}  →  {df['transit_day'].max().date()}")
print(f"\nCascade type distribution (% of all cascade days):")
for col in ["has_cash_cascade","has_gate_mech_in_chain","has_printer_in_chain",
            "has_csc_reader_in_chain","has_dopp_in_chain","has_scrst_in_chain"]:
    if col in df.columns:
        print(f"  {col:<30}: {df[col].mean():.2%}")
print(f"\nChain length stats:\n{df['chain_length'].describe().round(2)}")
print(f"\nData source: {DATA_SOURCE}")

In [ ]:
# ── CELL 5 : EDA ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

# Cascade type prevalence
type_cols = [c for c in ["has_cash_cascade","has_gate_mech_in_chain",
                          "has_printer_in_chain","has_csc_reader_in_chain",
                          "has_dopp_in_chain","has_scrst_in_chain"]
             if c in df.columns]
if type_cols:
    df[type_cols].mean().sort_values(ascending=False).plot(
        kind="bar", ax=axes[0,0], color="#1565C0", edgecolor="white")
    axes[0,0].set_title("Cascade type prevalence")
    axes[0,0].set_xticklabels(axes[0,0].get_xticklabels(), rotation=45, ha="right")
    axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda y,_: f"{y:.0%}"))

# Chain length distribution
df["chain_length"].plot(kind="hist", ax=axes[0,1], bins=20,
                        color="#43A047", edgecolor="white")
axes[0,1].set_title("Chain length distribution")
axes[0,1].set_xlabel("chain_length (faults per cascade day)")

# Cascade days over time
df.groupby("transit_day").size().rolling(7).mean().plot(
    ax=axes[0,2], color="#E53935", linewidth=0.8)
axes[0,2].set_title("Cascade days (7-day rolling avg)")

# Chain span distribution (minutes)
if "chain_span_min" in df.columns:
    df["chain_span_min"].clip(upper=120).plot(
        kind="hist", ax=axes[1,0], bins=30, color="#7B1FA2", edgecolor="white")
    axes[1,0].set_title("Chain span distribution (min, clipped @120)")
    axes[1,0].set_xlabel("chain_span_min")

# Device category breakdown
if "mars_device_category" in df.columns:
    df["mars_device_category"].value_counts().plot(
        kind="bar", ax=axes[1,1], color="#E65100", edgecolor="white")
    axes[1,1].set_title("Cascade days by device category")
    axes[1,1].set_xticklabels(axes[1,1].get_xticklabels(), rotation=0)

# Chain length by device category
if "mars_device_category" in df.columns:
    df.boxplot(column="chain_length", by="mars_device_category",
               ax=axes[1,2], patch_artist=True)
    axes[1,2].set_title("Chain length by device category")
    axes[1,2].set_xlabel("")
    plt.sca(axes[1,2]); plt.title("Chain length by device category")

plt.suptitle("PS2 — Cascade Chain EDA", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("/tmp/ps2_eda.png", dpi=100); plt.show()
print("EDA complete.")

In [ ]:
# ── CELL 6 : Feature engineering & temporal splits ───────────────────────────
# Analysis grain: (device_id, transit_day) cascade days only (chain_length >= 2).
# No prediction target — this is a pure analysis notebook.
# We derive a time-window bucket column from chain_span_min for downstream analysis.

SECONDARY_TARGETS = ["has_cash_cascade","has_gate_mech_in_chain",
                     "has_printer_in_chain","has_csc_reader_in_chain"]

EXCLUDE_COLS = [
    "DEVICE_ID","DEVICE_KEY","transit_day","mars_device_category",
    "DEVICE_NAME","FACILITY_ID","FACILITY_NAME","OPERATOR_ID","OPERATOR_NAME",
    "DEVICE_SERIAL_NUMBER","DEVICE_CONTROL_GROUP_TYPE_NAME","BUS_ID",
    "subsystem_chain","event_type_chain","severity_chain",
    "chain_start_dtm","chain_end_dtm",
    "first_subsystem","last_subsystem",
    "ci_related_devices","shared_facility_chain","svn_ci_data_available",
    "has_cascade",
] + SECONDARY_TARGETS

FEATURE_COLS = [
    c for c in df.columns
    if c not in EXCLUDE_COLS
    and pd.api.types.is_numeric_dtype(df[c])
]
df[FEATURE_COLS] = df[FEATURE_COLS].fillna(df[FEATURE_COLS].median())

# Temporal split (kept for Feature Store + HMM sequencing)
train_df = df[df["transit_day"] <  "2025-10-01"]
val_df   = df[(df["transit_day"] >= "2025-10-01") & (df["transit_day"] < "2026-01-01")]
test_df  = df[df["transit_day"] >= "2026-01-01"]

# Time-window bucket derived from chain_span_min
def assign_window(span):
    for label, (lo, hi) in TIME_WINDOWS.items():
        if lo <= span < hi:
            return label
    return "60+ min"

if "chain_span_min" in df.columns:
    df["window_bucket"] = df["chain_span_min"].apply(assign_window)
    window_order = list(TIME_WINDOWS.keys())
    df["window_bucket"] = pd.Categorical(df["window_bucket"],
                                          categories=window_order, ordered=True)

# Cascade velocity: average minutes per additional fault in chain
if "chain_span_min" in df.columns:
    df["cascade_velocity_min_per_fault"] = (
        df["chain_span_min"] / (df["chain_length"] - 1).clip(lower=1))

print(f"Features        : {len(FEATURE_COLS)}")
print(f"Train rows      : {len(train_df):,}  ({train_df['transit_day'].min().date()} → {train_df['transit_day'].max().date()})")
print(f"Val rows        : {len(val_df):,}  ({val_df['transit_day'].min().date()} → {val_df['transit_day'].max().date()})")
print(f"Test rows       : {len(test_df):,}  ({test_df['transit_day'].min().date()} → {test_df['transit_day'].max().date()})")

if "window_bucket" in df.columns:
    print(f"\n── Cascade time-window distribution ───────────────────────────────────")
    wc = df["window_bucket"].value_counts().sort_index()
    for w, n in wc.items():
        print(f"  {w:<12} : {n:>6,} cascade days  ({n/len(df):.1%})")

# ── Save checkpoint after feature engineering ─────────────────────────────
import os, joblib as _jl
_ckpt_dir = "/home/sagemaker-user/PS2/checkpoints"
os.makedirs(_ckpt_dir, exist_ok=True)
df.to_parquet(       os.path.join(_ckpt_dir, "df.parquet"),       index=True)
train_df.to_parquet( os.path.join(_ckpt_dir, "train_df.parquet"), index=True)
val_df.to_parquet(   os.path.join(_ckpt_dir, "val_df.parquet"),   index=True)
test_df.to_parquet(  os.path.join(_ckpt_dir, "test_df.parquet"),  index=True)
_jl.dump({
    "FEATURE_COLS":      FEATURE_COLS,
    "SECONDARY_TARGETS": SECONDARY_TARGETS,
    "EXCLUDE_COLS":      EXCLUDE_COLS,
}, os.path.join(_ckpt_dir, "meta.joblib"))
print(f"[CHECKPOINT] Saved to {_ckpt_dir}")
print(f"  df={df.shape}  train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")
print(f"  Next kernel restart: run Cells 1-3-3b only, then jump to Cell 10.")


In [ ]:
# ── CELL 7 : Feature Store (OPTIONAL — skip to CELL 9 for analysis) ──────────
# Feature Store is NOT required for any analysis cell (9-16).
# Enable only if you need online inference from the cascade features.
# WARNING: ingestion of 1.5M+ train rows takes 20-60 min.

ENABLE_FEATURE_STORE = False   # Set True only when online inference is needed

if not ENABLE_FEATURE_STORE:
    print("Feature Store skipped (ENABLE_FEATURE_STORE=False).")
    print("→ Continue to CELL 9 — all analysis cells work without Feature Store.")
else:
    _FS_EXCLUDE = {"chain_span_min", "cascade_velocity_min_per_fault", "window_bucket"}
    fs_cols = FEATURE_COLS + SECONDARY_TARGETS + ["DEVICE_ID", "transit_day"]
    fs_cols = [c for c in fs_cols if c in train_df.columns and c not in _FS_EXCLUDE]
    fs_df = train_df[fs_cols].copy()
    fs_df["event_time"] = fs_df["transit_day"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    fs_df["DEVICE_ID"]  = fs_df["DEVICE_ID"].astype(str)
    fs_df               = fs_df.drop(columns=["transit_day"])

    def dtype_to_fs(dtype):
        if pd.api.types.is_float_dtype(dtype):   return FeatureTypeEnum.FRACTIONAL
        if pd.api.types.is_integer_dtype(dtype): return FeatureTypeEnum.INTEGRAL
        return FeatureTypeEnum.STRING

    feature_group = FeatureGroup(
        name=FS_GROUP_NAME, sagemaker_session=SM_SESSION,
        feature_definitions=[
            FeatureDefinition(feature_name=c, feature_type=dtype_to_fs(fs_df[c].dtype))
            for c in fs_df.columns
        ])

    def fg_status():
        try:
            return feature_group.describe().get("FeatureGroupStatus")
        except Exception:
            return None

    try:
        st = fg_status()
        if st is not None:
            print(f"Existing Feature Group found (status: {st}) — deleting to recreate")
            feature_group.delete()
            import time as _time; _time.sleep(25)
        feature_group.create(
            s3_uri=f"s3://{ARTIFACT_BUCKET}/feature-store/{FS_GROUP_NAME}",
            record_identifier_name="DEVICE_ID", event_time_feature_name="event_time",
            role_arn=ROLE, enable_online_store=True,
            description="PS2 cascade chain features — Chicago Ventra")
        import time as _time
        while fg_status() != "Created":
            _time.sleep(15); print(f"  status: {fg_status()}")
        print(f"Feature Store group created: {FS_GROUP_NAME}  ({len(fs_df.columns)} features)")
    except Exception as e:
        print(f"Feature Group error: {e}")

In [ ]:
# ── CELL 8 : Ingest to Feature Store (OPTIONAL — gated by CELL 7 flag) ────────
if not ENABLE_FEATURE_STORE:
    print("Feature Store skipped — continuing to CELL 9.")
else:
    try:
        feature_group.ingest(data_frame=fs_df.reset_index(drop=True), max_workers=4, wait=True)
        print(f"Ingested {len(fs_df):,} records → {FS_GROUP_NAME}")
    except Exception as e:
        print(f"Feature Store ingest skipped: {str(e)[:150]}")

In [ ]:
# ── CELL 9 : Helper functions ─────────────────────────────────────────────────
COMMON_TAGS = {
    "ps"           : "PS2",
    "city"         : "chicago",
    "gold_version" : str(GOLD_VERSION),
    "architecture" : "CUBIC-MARS-AWS-v6",
    "analysis_type": "cascade_pattern_analysis",
}

def parse_chain(chain_str, sep="->"):
    """Parse 'BHU->CHU->PRINTER' into ['BHU','CHU','PRINTER']."""
    if pd.isna(chain_str) or not chain_str:
        return []
    return [s.strip() for s in str(chain_str).split(sep) if s.strip()]

def get_all_subsystems(df_sub, chain_col="subsystem_chain"):
    """Return sorted list of all unique subsystems across all chains."""
    all_subs = set()
    for chain in df_sub[chain_col].dropna():
        all_subs.update(parse_chain(chain))
    return sorted(all_subs)

def chain_to_binary_flags(df_sub, subsystems, chain_col="subsystem_chain"):
    """Return binary DataFrame: rows=cascade days, cols=subsystems (1 if present)."""
    rows = []
    for chain in df_sub[chain_col]:
        present = set(parse_chain(chain))
        rows.append({s: int(s in present) for s in subsystems})
    return pd.DataFrame(rows, index=df_sub.index)

def phi_corr(x, y):
    """Phi (Matthews) correlation coefficient for two binary series."""
    n   = len(x)
    n11 = (x & y).sum(); n10 = (x & ~y).sum()
    n01 = (~x & y).sum(); n00 = (~x & ~y).sum()
    n1_ = n11 + n10; n0_ = n01 + n00
    n_1 = n11 + n01; n_0 = n10 + n00
    denom = np.sqrt(n1_ * n0_ * n_1 * n_0)
    return (n * n11 - n1_ * n_1) / denom if denom > 0 else 0.0

def save_and_log(fig, path, run=None):
    """Save figure and optionally log to active MLflow run."""
    fig.savefig(path, dpi=100, bbox_inches="tight")
    plt.show(); plt.close(fig)
    if run is not None:
        mlflow.log_artifact(path)
    return path

In [ ]:
# ── CELL 10 : Cascade Propagation — Time-Window Analysis ─────────────────────
if "window_bucket" not in df.columns:
    print("chain_span_min not available — time-window analysis skipped")
else:
    with mlflow.start_run(run_name="cascade_time_window_analysis") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "time_window_analysis"})
        mlflow.log_param("time_windows", str(list(TIME_WINDOWS.keys())))
        window_order = list(TIME_WINDOWS.keys())

        # ── 1. Chain length by window ─────────────────────────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        wg = df.groupby("window_bucket", observed=True)["chain_length"]
        wg.mean().reindex(window_order).plot(kind="bar", ax=axes[0],
            color="#1565C0", edgecolor="white")
        axes[0].set_title("Mean chain length per time window")
        axes[0].set_xlabel(""); axes[0].set_ylabel("Avg faults in cascade")
        axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha="right")
        df.boxplot(column="chain_length", by="window_bucket",
                   ax=axes[1], patch_artist=True,
                   positions=range(len(window_order)))
        axes[1].set_xticklabels(window_order, rotation=30, ha="right")
        plt.sca(axes[1]); plt.title("Chain length distribution by time window")
        plt.suptitle("Cascade Propagation — Fault Count vs Speed", fontsize=11)
        plt.tight_layout()
        save_and_log(fig, "/tmp/ps2_window_chain_length.png", run)

        # ── 2. Cascade velocity by window and device category ─────────────────
        if "cascade_velocity_min_per_fault" in df.columns and "mars_device_category" in df.columns:
            fig, ax = plt.subplots(figsize=(12, 5))
            vel_pivot = (df.groupby(["window_bucket","mars_device_category"],
                                     observed=True)["cascade_velocity_min_per_fault"]
                           .mean().unstack("mars_device_category").reindex(window_order))
            vel_pivot.plot(kind="bar", ax=ax, edgecolor="white")
            ax.set_title("Cascade velocity (min/fault) by time window and device category")
            ax.set_xlabel("Time window"); ax.set_ylabel("Min per additional fault")
            ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
            ax.legend(title="Device category")
            plt.tight_layout()
            save_and_log(fig, "/tmp/ps2_cascade_velocity.png", run)

        # ── 3. Window distribution by device category ─────────────────────────
        if "mars_device_category" in df.columns:
            fig, ax = plt.subplots(figsize=(10, 5))
            cat_window = (df.groupby(["mars_device_category","window_bucket"], observed=True)
                            .size().unstack("window_bucket", fill_value=0)
                            .reindex(columns=window_order, fill_value=0))
            cat_window_pct = cat_window.div(cat_window.sum(axis=1), axis=0) * 100
            cat_window_pct.plot(kind="bar", stacked=True, ax=ax, edgecolor="white")
            ax.set_title("Cascade speed regime distribution by device category (%)")
            ax.set_xlabel("Device category"); ax.set_ylabel("% of cascade days")
            ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
            ax.legend(title="Time window", bbox_to_anchor=(1.05,1), loc="upper left")
            plt.tight_layout()
            save_and_log(fig, "/tmp/ps2_window_by_device_cat.png", run)

        # ── 4. Summary stats table ────────────────────────────────────────────
        agg_cols = {"chain_length": ["mean","median","max"],
                    "chain_span_min": ["mean","median"],
                    "cascade_velocity_min_per_fault": ["mean"]}
        agg_cols = {k: v for k, v in agg_cols.items() if k in df.columns}
        summary = df.groupby("window_bucket", observed=True).agg(
            count=("chain_length","count"), **{
                f"{col}_{stat}": (col, stat)
                for col, stats_list in agg_cols.items() for stat in stats_list
            }).reindex(window_order)
        summary.to_csv("/tmp/ps2_window_summary.csv")
        mlflow.log_artifact("/tmp/ps2_window_summary.csv")
        print("\n── Cascade Time-Window Summary ──────────────────────────────────────")
        print(summary.to_string())
        print("\n── Window Interpretation ────────────────────────────────────────────")
        print(f"  60+ min dominates: {summary.loc['60+ min','count']:,} cascade days "
              f"({100*summary.loc['60+ min','count']/summary['count'].sum():.1f}%)")
        print(f"  Slow cascades (60+) have {summary.loc['60+ min','chain_length_mean']:.1f}× "
              f"more faults than fast (0-5 min): "
              f"{summary.loc['0-5 min','chain_length_mean']:.1f} faults avg")
        if "chain_span_min_mean" in summary.columns:
            ratio = (summary.loc["60+ min","chain_span_min_mean"] /
                     max(summary.loc["0-5 min","chain_span_min_mean"], 0.001))
            print(f"  Span ratio 60+/0-5: {ratio:.0f}× longer")

        def _mk(w): return w.replace(' ','_').replace('+','p').replace('-','to')
        for w in window_order:
            if w in summary.index:
                mlflow.log_metric(f"window_{_mk(w)}_count", int(summary.loc[w,"count"]))
                if "chain_length_mean" in summary.columns:
                    mlflow.log_metric(f"window_{_mk(w)}_avg_chain_len",
                                      round(float(summary.loc[w,"chain_length_mean"]),3))

        # ── 5. Top 20 most cascade-active devices ─────────────────────────────
        top_dev_counts = df.groupby("DEVICE_ID").size().nlargest(20).rename("cascade_days")
        top_dev_df = top_dev_counts.to_frame()
        if "mars_device_category" in df.columns:
            dev_cat = (df.drop_duplicates("DEVICE_ID")
                        .set_index("DEVICE_ID")["mars_device_category"])
            top_dev_df["category"] = top_dev_df.index.map(dev_cat)
        if "window_bucket" in df.columns:
            for w in window_order:
                df_w = df[df["window_bucket"] == w]
                wc = df_w.groupby("DEVICE_ID").size().rename(w)
                top_dev_df[w] = top_dev_df.index.map(wc).fillna(0).astype(int)
        top_dev_df.to_csv("/tmp/ps2_top20_devices.csv")
        mlflow.log_artifact("/tmp/ps2_top20_devices.csv")
        print(f"\n── Top 20 Most Cascade-Active Devices ──────────────────────────────────")
        print(top_dev_df.to_string())
        mlflow.log_metric("top_device_cascade_days", int(top_dev_counts.iloc[0]))

        # ── 6. Device × time-window heatmap (top 10 devices) ─────────────────
        top10_ids = top_dev_counts.index[:10].tolist()
        if "window_bucket" in df.columns:
            dw = (df[df["DEVICE_ID"].isin(top10_ids)]
                  .groupby(["DEVICE_ID","window_bucket"], observed=True)
                  .size().unstack("window_bucket", fill_value=0)
                  .reindex(columns=window_order, fill_value=0))
            fig, ax = plt.subplots(figsize=(13, 6))
            sns.heatmap(dw, cmap="YlOrRd", annot=True, fmt="d",
                        ax=ax, linewidths=0.4, linecolor="white",
                        cbar_kws={"label": "Cascade days"})
            ax.set_title("Top-10 Devices: Cascade Days per Time Window\n"
                         "(each row = one device, each col = speed regime)")
            ax.set_xlabel("Time Window"); ax.set_ylabel("Device ID")
            ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
            ax.tick_params(axis="y", labelsize=8)
            plt.tight_layout()
            save_and_log(fig, "/tmp/ps2_top10_device_window_heatmap.png", run)

        # ── 7. Top device deep-dive ───────────────────────────────────────────
        top_dev_id = top10_ids[0]
        df_top = df[df["DEVICE_ID"] == top_dev_id]
        print(f"\n── Top Device Deep-Dive: {top_dev_id} ────────────────────────────────")
        print(f"  Cascade days   : {len(df_top):,}")
        if "mars_device_category" in df.columns:
            print(f"  Category       : {df_top['mars_device_category'].iloc[0]}")
        print(f"  Date range     : {df_top['transit_day'].min().date()} → "
              f"{df_top['transit_day'].max().date()}")
        print(f"  Chain length   : mean={df_top['chain_length'].mean():.2f}  "
              f"median={df_top['chain_length'].median():.0f}  "
              f"max={df_top['chain_length'].max()}")
        if "chain_span_min" in df_top.columns:
            print(f"  Span (min)     : mean={df_top['chain_span_min'].mean():.1f}  "
                  f"max={df_top['chain_span_min'].max():.1f}")
        if "window_bucket" in df_top.columns:
            wc_top = df_top["window_bucket"].value_counts().sort_index()
            print(f"  Time windows:\n"
                  + "\n".join(f"    {w:<12}: {n:>6,}" for w, n in wc_top.items()))
        # Monthly trend for top device
        monthly = df_top.set_index("transit_day").resample("ME").size().rename("n")
        print(f"\n  Monthly cascade days (all months):")
        for dt, n in monthly.items():
            bar = "█" * min(int(n / max(monthly.max(),1) * 20), 20)
            print(f"    {dt.strftime('%Y-%m')} : {n:>5,}  {bar}")

        print("\nTime-window analysis logged to MLflow.")

In [ ]:
# ── CELL 11 : Subsystem φ (Phi) Correlation Matrix ───────────────────────────
# Phi = Matthews Correlation Coefficient for two binary variables.
# For each subsystem pair: does A failing predict B failing in the same chain?
# Computed overall, per device category, AND for top-10 individual devices.

if "subsystem_chain" not in df.columns:
    print("subsystem_chain not available — phi correlation skipped")
else:
    with mlflow.start_run(run_name="subsystem_phi_correlation") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "phi_correlation_matrix"})

        def compute_phi_matrix(df_sub, label="all"):
            subsystems = get_all_subsystems(df_sub)
            if len(subsystems) < 2:
                print(f"  [{label}] fewer than 2 subsystems — skipped")
                return None, subsystems
            flags = chain_to_binary_flags(df_sub, subsystems).astype(bool)
            phi_mat = pd.DataFrame(np.zeros((len(subsystems), len(subsystems))),
                                   index=subsystems, columns=subsystems)
            for s1, s2 in combinations(subsystems, 2):
                p = phi_corr(flags[s1], flags[s2])
                phi_mat.loc[s1, s2] = p
                phi_mat.loc[s2, s1] = p
            np.fill_diagonal(phi_mat.values, 1.0)
            return phi_mat, subsystems

        def plot_phi_matrix(phi_mat, title, path):
            fig, ax = plt.subplots(figsize=(max(9, len(phi_mat)*1.0),
                                            max(7, len(phi_mat)*0.9)))
            sns.heatmap(phi_mat, annot=True, fmt=".2f",
                        cmap="RdYlGn", center=0,
                        ax=ax, linewidths=0.4, square=True,
                        annot_kws={"size": 8})
            ax.set_title(title, fontsize=11)
            ax.tick_params(axis="x", rotation=45, labelsize=8)
            ax.tick_params(axis="y", rotation=0, labelsize=8)
            plt.tight_layout()
            save_and_log(fig, path, run)

        # ── Overall phi matrix ────────────────────────────────────────────────
        phi_all, subs_all = compute_phi_matrix(df, label="overall")
        if phi_all is not None:
            plot_phi_matrix(phi_all,
                "PS2 — Subsystem Phi Correlation (all devices, all cascade days)",
                "/tmp/ps2_phi_overall.png")
            phi_all.to_csv("/tmp/ps2_phi_overall.csv")
            mlflow.log_artifact("/tmp/ps2_phi_overall.csv")

            phi_upper = phi_all.where(
                np.triu(np.ones(phi_all.shape), k=1).astype(bool))
            top_pairs = (phi_upper.stack()
                         .reset_index()
                         .rename(columns={"level_0":"sub_a","level_1":"sub_b",0:"phi"})
                         .sort_values("phi", ascending=False))
            top_pairs.to_csv("/tmp/ps2_phi_pairs.csv", index=False)
            mlflow.log_artifact("/tmp/ps2_phi_pairs.csv")

            print("\n── Top 10 Positively Correlated Subsystem Pairs (phi) ──────────────────")
            print(top_pairs.head(10).to_string(index=False))
            for _, r in top_pairs.head(10).iterrows():
                key = f"phi_{r['sub_a']}__{r['sub_b']}".replace(" ","_")
                mlflow.log_metric(key, round(float(r["phi"]), 4))

            # Bottom 5 (negatively correlated)
            bottom_pairs = top_pairs.tail(5).sort_values("phi")
            print("\n── Bottom 5 Negatively Correlated Subsystem Pairs (phi) ───────────────")
            print(bottom_pairs.to_string(index=False))

        # ── Per device category ───────────────────────────────────────────────
        if "mars_device_category" in df.columns:
            for cat in df["mars_device_category"].dropna().unique():
                df_cat = df[df["mars_device_category"] == cat]
                phi_cat, _ = compute_phi_matrix(df_cat, label=cat)
                if phi_cat is not None:
                    plot_phi_matrix(phi_cat,
                        f"PS2 — Subsystem Phi Correlation [{cat}]",
                        f"/tmp/ps2_phi_{cat.lower()}.png")
                    phi_cat.to_csv(f"/tmp/ps2_phi_{cat.lower()}.csv")
                    mlflow.log_artifact(f"/tmp/ps2_phi_{cat.lower()}.csv")
                    # Top pair for this category
                    pu = phi_cat.where(np.triu(np.ones(phi_cat.shape), k=1).astype(bool))
                    tp = (pu.stack().reset_index()
                           .rename(columns={"level_0":"sub_a","level_1":"sub_b",0:"phi"})
                           .sort_values("phi", ascending=False))
                    if len(tp):
                        r = tp.iloc[0]
                        print(f"  [{cat}] top pair: {r['sub_a']} ↔ {r['sub_b']}  phi={r['phi']:.3f}")
                        print(f"         top 5:")
                        print(tp.head(5).to_string(index=False))

        # ── Per individual device — top 10 most cascade-active ────────────────
        top_dev_counts = df.groupby("DEVICE_ID").size().nlargest(10)
        print(f"\n── Per-Device Phi Correlation (Top 10 Cascade-Active Devices) ──────────")
        dev_phi_summary = []
        for dev_id, n_cascade in top_dev_counts.items():
            df_dev = df[df["DEVICE_ID"] == dev_id]
            if len(df_dev) < 10:
                print(f"  {dev_id}: only {len(df_dev)} cascade days — skipped")
                continue
            phi_dev, _ = compute_phi_matrix(df_dev, label=str(dev_id))
            if phi_dev is None:
                continue
            pu = phi_dev.where(np.triu(np.ones(phi_dev.shape), k=1).astype(bool))
            tp_dev = (pu.stack().reset_index()
                       .rename(columns={"level_0":"sub_a","level_1":"sub_b",0:"phi"})
                       .sort_values("phi", ascending=False))
            cat_label = ""
            if "mars_device_category" in df.columns:
                cat_vals = df_dev["mars_device_category"].dropna()
                cat_label = f" [{cat_vals.iloc[0]}]" if len(cat_vals) else ""
            print(f"\n  Device {dev_id}{cat_label}  ({n_cascade} cascade days):")
            if len(tp_dev):
                print(f"    Top 5 correlated pairs:")
                print(tp_dev.head(5).to_string(index=False))
                dev_phi_summary.append({
                    "DEVICE_ID": dev_id,
                    "cascade_days": n_cascade,
                    "top_pair": f"{tp_dev.iloc[0]['sub_a']} ↔ {tp_dev.iloc[0]['sub_b']}",
                    "top_phi": round(tp_dev.iloc[0]["phi"], 3),
                })
            # Plot phi heatmap for top device
            if dev_id == top_dev_counts.index[0]:
                phi_dev.to_csv("/tmp/ps2_phi_topdevice.csv")
                mlflow.log_artifact("/tmp/ps2_phi_topdevice.csv")
                plot_phi_matrix(phi_dev,
                    f"PS2 — Phi Correlation: Top Device ({dev_id}, {n_cascade} cascade days)",
                    "/tmp/ps2_phi_topdevice.png")

        if dev_phi_summary:
            summ_df = pd.DataFrame(dev_phi_summary)
            summ_df.to_csv("/tmp/ps2_per_device_phi_summary.csv", index=False)
            mlflow.log_artifact("/tmp/ps2_per_device_phi_summary.csv")
            print(f"\n── Per-Device Phi Summary ──────────────────────────────────────────────")
            print(summ_df.to_string(index=False))

        print("\nPhi correlation analysis logged to MLflow.")

In [ ]:
# ── CELL 12 : Conditional Probability P(B | A, T) ────────────────────────────
# P(B|A,T) = fraction of chains in window T that contain BOTH A and B,
#             among all chains in window T that contain A.
# Layout: 2×3 grid (one large heatmap per time window) + per-device analysis.

if "subsystem_chain" not in df.columns or "window_bucket" not in df.columns:
    print("subsystem_chain or window_bucket not available — conditional prob skipped")
else:
    with mlflow.start_run(run_name="conditional_probability_PBgivenAT") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "conditional_probability_PBgivenAT"})

        subsystems   = get_all_subsystems(df)
        window_order = list(TIME_WINDOWS.keys())
        n_sub        = len(subsystems)

        # ── Build per-window P(B|A) matrices ─────────────────────────────────
        pb_given_a = {}
        window_sizes = {}
        for w in window_order:
            df_w = df[df["window_bucket"] == w]
            window_sizes[w] = len(df_w)
            if len(df_w) == 0:
                pb_given_a[w] = pd.DataFrame(np.zeros((n_sub, n_sub)),
                                              index=subsystems, columns=subsystems)
                continue
            flags_w = chain_to_binary_flags(df_w, subsystems).astype(bool)
            mat = pd.DataFrame(np.zeros((n_sub, n_sub)),
                               index=subsystems, columns=subsystems)
            for i, sub_a in enumerate(subsystems):
                mask_a = flags_w[sub_a]
                n_a    = mask_a.sum()
                if n_a == 0:
                    continue
                for j, sub_b in enumerate(subsystems):
                    mat.iloc[i, j] = (1.0 if i == j
                                      else (mask_a & flags_w[sub_b]).sum() / n_a)
            pb_given_a[w] = mat

        # Save combined CSV
        all_rows = []
        for w, mat in pb_given_a.items():
            tmp = mat.stack().reset_index()
            tmp.columns = ["sub_a","sub_b","P_B_given_A"]
            tmp["window"] = w
            all_rows.append(tmp)
        cond_df = pd.concat(all_rows, ignore_index=True)
        cond_df.to_csv("/tmp/ps2_cond_prob.csv", index=False)
        mlflow.log_artifact("/tmp/ps2_cond_prob.csv")

        # ── Plot: 2×3 grid — one large heatmap per time window ───────────────
        n_windows = len(window_order)
        nrows, ncols = 2, 3
        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(ncols * 7, nrows * max(5, n_sub * 0.65)))
        axes_flat = axes.flatten()

        for idx, w in enumerate(window_order):
            ax  = axes_flat[idx]
            mat = pb_given_a[w]
            n_w = window_sizes[w]
            annot = n_sub <= 12   # annotate only when not too many subsystems
            sns.heatmap(mat, annot=annot, fmt=".2f",
                        cmap="Blues", vmin=0, vmax=1,
                        ax=ax, cbar=True,
                        xticklabels=True, yticklabels=True,
                        linewidths=0.3, linecolor="lightgray",
                        annot_kws={"size": 8})
            ax.set_title(f"{w}  (n={n_w:,})", fontsize=11, pad=8)
            ax.set_xlabel("→ B (consequent)", fontsize=9)
            ax.set_ylabel("← A (antecedent)", fontsize=9)
            ax.tick_params(axis="x", rotation=45, labelsize=8)
            ax.tick_params(axis="y", rotation=0,  labelsize=8)

        for ax in axes_flat[n_windows:]:
            ax.set_visible(False)

        fig.suptitle("P(B | A, T) — Conditional probability subsystem B fails given A fails\n"
                     "(rows = A, cols = B, each subplot = time window)",
                     fontsize=13, y=1.01)
        plt.tight_layout()
        save_and_log(fig, "/tmp/ps2_cond_prob_grid.png", run)

        # ── Top 10 conditional pairs across all windows ───────────────────────
        top_cond = (cond_df[cond_df["sub_a"] != cond_df["sub_b"]]
                    .sort_values("P_B_given_A", ascending=False)
                    .head(10))
        print("\n── Top 10 Conditional Probability Pairs P(B|A,T) ──────────────────────")
        print(top_cond.to_string(index=False))
        for _, r in top_cond.iterrows():
            key = (f"P_{r['sub_b']}_given_{r['sub_a']}_"
                   f"{r['window'].replace(' ','_').replace('+','p').replace('-','to')}")
            mlflow.log_metric(key, round(float(r["P_B_given_A"]), 4))

        # ── Summary: highest average P(B|A) across all windows ───────────────
        avg_pba = (cond_df[cond_df["sub_a"] != cond_df["sub_b"]]
                   .groupby(["sub_a","sub_b"])["P_B_given_A"].mean()
                   .reset_index().sort_values("P_B_given_A", ascending=False))
        print("\n── Top 10 Pairs by Average P(B|A) Across All Time Windows ────────────")
        print(avg_pba.head(10).to_string(index=False))

        # ── Per-device: top device ────────────────────────────────────────────
        top_dev_id = df.groupby("DEVICE_ID").size().idxmax()
        df_dev = df[df["DEVICE_ID"] == top_dev_id]
        n_dev  = len(df_dev)
        print(f"\n── P(B|A) for Top Device: {top_dev_id}  ({n_dev:,} cascade days) ────────")
        if n_dev >= 10 and "window_bucket" in df_dev.columns:
            dev_subs = get_all_subsystems(df_dev)
            if len(dev_subs) >= 2:
                dev_flags = chain_to_binary_flags(df_dev, dev_subs).astype(bool)
                dev_mat   = pd.DataFrame(np.zeros((len(dev_subs), len(dev_subs))),
                                         index=dev_subs, columns=dev_subs)
                for i, sa in enumerate(dev_subs):
                    mask_a = dev_flags[sa]; n_a = mask_a.sum()
                    if n_a == 0: continue
                    for j, sb in enumerate(dev_subs):
                        dev_mat.iloc[i, j] = (1.0 if i == j
                                               else (mask_a & dev_flags[sb]).sum() / n_a)
                # Print top pairs for this device
                dev_upper = dev_mat.where(np.triu(np.ones(dev_mat.shape), k=1).astype(bool))
                dev_top = (dev_upper.stack().reset_index()
                           .rename(columns={"level_0":"sub_a","level_1":"sub_b",0:"P_B_given_A"})
                           .sort_values("P_B_given_A", ascending=False))
                print(f"  Top 10 subsystem pairs by P(B|A):")
                print(dev_top.head(10).to_string(index=False))
                # Plot overall P(B|A) heatmap for top device
                fig_d, ax_d = plt.subplots(figsize=(max(8, len(dev_subs)*0.9),
                                                     max(7, len(dev_subs)*0.8)))
                sns.heatmap(dev_mat, annot=True, fmt=".2f",
                            cmap="Blues", vmin=0, vmax=1,
                            ax=ax_d, linewidths=0.4, annot_kws={"size": 8})
                ax_d.set_title(f"P(B|A) — Top Device: {top_dev_id}  ({n_dev:,} cascade days)")
                ax_d.tick_params(axis="x", rotation=45, labelsize=8)
                ax_d.tick_params(axis="y", rotation=0, labelsize=8)
                plt.tight_layout()
                save_and_log(fig_d, "/tmp/ps2_cond_prob_topdevice.png", run)
                dev_mat.to_csv("/tmp/ps2_cond_prob_topdevice.csv")
                mlflow.log_artifact("/tmp/ps2_cond_prob_topdevice.csv")
        else:
            print(f"  Device has only {n_dev} cascade days — P(B|A) unreliable")

        print("\nConditional probability analysis logged to MLflow.")

In [ ]:
# ── CELL 13 : Markov Chain — cascade transition probabilities ─────────────────
# Builds a subsystem-to-subsystem transition probability matrix overall AND
# per device category AND for top-20 individual devices.
# Gold grain = cascade days only — all rows are cascades.

def build_markov(df_sub, chain_col="subsystem_chain"):
    chains = (df_sub[chain_col]
              .dropna()
              .apply(lambda s: [x.strip() for x in str(s).split("->") if x.strip()])
              .tolist())
    chains = [c for c in chains if len(c) >= 2]
    if not chains:
        return None, [], 0
    states = sorted(set(s for c in chains for s in c))
    idx    = {s: i for i, s in enumerate(states)}
    n      = len(states)
    trans  = np.zeros((n, n))
    for chain in chains:
        for a, b in zip(chain[:-1], chain[1:]):
            if a in idx and b in idx:
                trans[idx[a], idx[b]] += 1
    row_sums = trans.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return trans / row_sums, states, len(chains)

if "subsystem_chain" not in df.columns:
    print("subsystem_chain not in Gold — Markov analysis skipped")
    MARKOV_RUN = None
else:
    with mlflow.start_run(run_name="markov_chain_analysis") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "markov_chain",
                         "stage": "analysis"})

        # ── 1. Overall transition matrix ──────────────────────────────────────
        tp_all, states_all, n_chains_all = build_markov(train_df)
        if tp_all is not None:
            trans_df_all = pd.DataFrame(tp_all, index=states_all, columns=states_all)
            trans_df_all.to_csv("/tmp/markov_transition_all.csv")
            mlflow.log_artifact("/tmp/markov_transition_all.csv")
            mlflow.log_param("n_states_overall",  len(states_all))
            mlflow.log_param("n_chains_overall",  n_chains_all)

            fig, ax = plt.subplots(figsize=(max(8, len(states_all)*0.9),
                                             max(6, len(states_all)*0.8)))
            sns.heatmap(trans_df_all, annot=True, fmt=".2f",
                        cmap="YlOrRd", ax=ax, linewidths=0.5,
                        annot_kws={"size": 8})
            ax.set_title("PS2 Markov Chain — Subsystem Cascade Transition P\n"
                         "(all devices, train split)")
            plt.tight_layout()
            save_and_log(fig, "/tmp/markov_transition_all.png", run)

            print("\n── Overall Markov Chain Transition Summary ──────────────────────────────")
            print(f"  States : {states_all}")
            print(f"  Chains : {n_chains_all:,}")
            high_trans = [(states_all[i], states_all[j], tp_all[i,j])
                          for i in range(len(states_all))
                          for j in range(len(states_all))
                          if tp_all[i,j] > 0.1]
            high_trans.sort(key=lambda x: -x[2])
            print("\n  All transitions with P > 10%:")
            for s1, s2, p in high_trans:
                bar = "█" * int(p * 20)
                self_lbl = " (self-loop)" if s1 == s2 else ""
                print(f"    {s1:<15} → {s2:<15}  {p:.1%}  {bar}{self_lbl}")
            for s1, s2, p in high_trans[:10]:
                mlflow.log_metric(f"trans_{s1}_to_{s2}", round(p, 3))

        # ── 2. Per device category ────────────────────────────────────────────
        if "mars_device_category" in df.columns:
            cats = train_df["mars_device_category"].dropna().unique()
            n_cats = len(cats)
            if n_cats > 0:
                fig_cat, axes_cat = plt.subplots(
                    1, n_cats,
                    figsize=(max(7, len(states_all)*0.8) * n_cats,
                             max(5, len(states_all)*0.7)),
                    squeeze=False)
                for col_idx, cat in enumerate(sorted(cats)):
                    df_cat = train_df[train_df["mars_device_category"] == cat]
                    tp_cat, states_cat, n_chains_cat = build_markov(df_cat)
                    if tp_cat is None:
                        axes_cat[0,col_idx].set_title(f"[{cat}] — no data"); continue
                    trans_df_cat = pd.DataFrame(tp_cat, index=states_cat, columns=states_cat)
                    trans_df_cat.to_csv(f"/tmp/markov_transition_{cat.lower()}.csv")
                    mlflow.log_artifact(f"/tmp/markov_transition_{cat.lower()}.csv")
                    mlflow.log_param(f"n_chains_{cat}", n_chains_cat)
                    sns.heatmap(trans_df_cat, annot=True, fmt=".2f",
                                cmap="YlOrRd", ax=axes_cat[0,col_idx],
                                linewidths=0.4, cbar=False,
                                annot_kws={"size": 7})
                    axes_cat[0,col_idx].set_title(f"[{cat}] ({n_chains_cat:,} chains)",
                                                   fontsize=10)
                    axes_cat[0,col_idx].tick_params(axis="x", rotation=90, labelsize=7)
                    axes_cat[0,col_idx].tick_params(axis="y", labelsize=7)
                fig_cat.suptitle("PS2 Markov — Transition Prob by Device Category",
                                 fontsize=11)
                plt.tight_layout()
                save_and_log(fig_cat, "/tmp/markov_transition_by_cat.png", run)

        # ── 3. Top cascade paths (overall) ───────────────────────────────────
        all_chains = (train_df["subsystem_chain"]
                      .dropna()
                      .apply(lambda s: [x.strip() for x in str(s).split("->")
                                        if x.strip()])
                      .tolist())
        path_counts = Counter(["->".join(c) for c in all_chains if len(c) >= 2])
        top_paths   = pd.DataFrame(path_counts.most_common(20),
                                   columns=["path","count"])
        top_paths.to_csv("/tmp/top_cascade_paths.csv", index=False)
        mlflow.log_artifact("/tmp/top_cascade_paths.csv")
        print(f"\n── Top 20 Cascade Paths (overall, train split) ──────────────────────────")
        print(top_paths.to_string(index=False))

        # ── 4. Top 20 most cascade-active individual devices ─────────────────
        top_dev_counts = train_df.groupby("DEVICE_ID").size().nlargest(20).rename("cascade_days")
        top_dev_info = top_dev_counts.to_frame()
        if "mars_device_category" in train_df.columns:
            dev_cat = (train_df.drop_duplicates("DEVICE_ID")
                        .set_index("DEVICE_ID")["mars_device_category"])
            top_dev_info["category"] = top_dev_info.index.map(dev_cat)
        top_dev_info.to_csv("/tmp/ps2_top20_devices_markov.csv")
        mlflow.log_artifact("/tmp/ps2_top20_devices_markov.csv")
        print(f"\n── Top 20 Most Cascade-Active Devices (train split) ─────────────────────")
        print(top_dev_info.to_string())
        mlflow.log_metric("top_device_cascade_days_train", int(top_dev_counts.iloc[0]))

        # ── 5. Per-device Markov + top paths for top 5 devices ───────────────
        print(f"\n── Per-Device Markov Analysis (Top 5 Devices) ───────────────────────────")
        for rank, (dev_id, n_cascade) in enumerate(top_dev_counts.head(5).items()):
            df_dev = train_df[train_df["DEVICE_ID"] == dev_id]
            cat_lbl = ""
            if "mars_device_category" in df_dev.columns:
                c = df_dev["mars_device_category"].dropna()
                cat_lbl = f" [{c.iloc[0]}]" if len(c) else ""
            print(f"\n  ── Device #{rank+1}: {dev_id}{cat_lbl}  "
                  f"({n_cascade} cascade days) ──────────────────────")

            tp_dev, states_dev, n_chains_dev = build_markov(df_dev)
            if tp_dev is None:
                print("    No multi-subsystem chains — skipped")
                continue

            # Top cascade paths for this device
            dev_chains_raw = (df_dev["subsystem_chain"]
                              .dropna()
                              .apply(lambda s: [x.strip() for x in str(s).split("->")
                                                if x.strip()])
                              .tolist())
            dev_path_counts = Counter(["->".join(c) for c in dev_chains_raw
                                       if len(c) >= 2])
            dev_top_paths = dev_path_counts.most_common(10)
            print(f"    Top 10 cascade paths ({n_chains_dev} total chains):")
            for path_str, cnt in dev_top_paths:
                print(f"      {path_str:<60}  count={cnt:>6,}")

            # Highest transition probabilities
            trans_df_dev = pd.DataFrame(tp_dev, index=states_dev, columns=states_dev)
            high_dev = [(states_dev[i], states_dev[j], tp_dev[i,j])
                        for i in range(len(states_dev))
                        for j in range(len(states_dev))
                        if tp_dev[i,j] > 0.15]
            high_dev.sort(key=lambda x: -x[2])
            print(f"    Top transitions (P > 15%):")
            for s1, s2, p in high_dev[:8]:
                self_lbl = " (self-loop)" if s1 == s2 else ""
                print(f"      {s1:<15} → {s2:<15}  {p:.1%}{self_lbl}")

            # Save + plot for top device
            if rank == 0:
                trans_df_dev.to_csv("/tmp/markov_transition_topdevice.csv")
                mlflow.log_artifact("/tmp/markov_transition_topdevice.csv")
                fig_d, ax_d = plt.subplots(
                    figsize=(max(7, len(states_dev)*0.9),
                             max(5, len(states_dev)*0.8)))
                sns.heatmap(trans_df_dev, annot=True, fmt=".2f",
                            cmap="YlOrRd", ax=ax_d, linewidths=0.5,
                            annot_kws={"size": 9})
                ax_d.set_title(f"Markov Chain: Top Device {dev_id}{cat_lbl}\n"
                               f"({n_cascade} cascade days, {n_chains_dev} chains)")
                plt.tight_layout()
                save_and_log(fig_d, "/tmp/markov_transition_topdevice.png", run)

        MARKOV_RUN = run.info.run_id
    print(f"\nMarkov analysis logged (run={MARKOV_RUN})")

In [ ]:
# ── CELL 14 : HMM — 3-state Gaussian HMM on per-device cascade severity ───────
# Features: chain_length, critical_in_chain, oos_in_chain,
#            chain_span_min, distinct_subsystems.
# Grain: (device_id, transit_day) cascade days sorted by date.
# States auto-sorted by chain_length mean → Minor / Moderate / Critical.

HMM_FEATURES = [c for c in ["chain_length", "critical_in_chain", "oos_in_chain",
                              "chain_span_min", "distinct_subsystems"]
                 if c in df.columns]

if len(HMM_FEATURES) < 2:
    print("Insufficient HMM features in Gold — skipped")
    HMM_RUN = None
else:
    hmm_data = (train_df.sort_values(["DEVICE_ID", "transit_day"])
                         .groupby("DEVICE_ID")[HMM_FEATURES]
                         .apply(lambda x: x.values.tolist())
                         .reset_index()
                         .rename(columns={0: "sequence"}))

    sequences = [np.array(seq) for seq in hmm_data["sequence"] if len(seq) >= 3]
    if not sequences:
        print("No device has >= 3 cascade days in train split — HMM skipped")
        HMM_RUN = None
    else:
        X_hmm   = np.vstack(sequences)
        lengths = [len(s) for s in sequences]

        model_hmm = hmm.GaussianHMM(
            n_components=3, covariance_type="diag",
            n_iter=150, random_state=42, verbose=False)
        model_hmm.fit(X_hmm, lengths)

        # Auto-sort states: 0=Minor, 1=Moderate, 2=Critical by chain_length mean
        cl_idx = HMM_FEATURES.index("chain_length") if "chain_length" in HMM_FEATURES else 0
        state_order = np.argsort(model_hmm.means_[:, cl_idx])
        STATE_NAMES = {state_order[0]: "Minor cascade",
                       state_order[1]: "Moderate cascade",
                       state_order[2]: "Critical cascade"}
        STATE_LABELS = [STATE_NAMES[i] for i in range(3)]

        # Decode Viterbi path for ALL sequences to get state prevalence
        all_states = []
        idx_start  = 0
        for seq in sequences:
            _, path = model_hmm.decode(seq, algorithm="viterbi")
            all_states.extend(path.tolist())
            idx_start += len(seq)
        state_counts = Counter(all_states)
        total_obs    = sum(state_counts.values())

        # Dwell time: expected consecutive days in state = 1 / (1 - P_self)
        dwell_times = {i: (1.0 / max(1.0 - model_hmm.transmat_[i, i], 1e-9))
                       for i in range(3)}

        # Build re-ordered transition matrix for display
        trans_mat_df = pd.DataFrame(
            model_hmm.transmat_,
            columns=[f"→S{i}({STATE_NAMES[i][:8]})" for i in range(3)],
            index=[f"S{i}({STATE_NAMES[i][:8]})" for i in range(3)])

        with mlflow.start_run(run_name="hmm_3state") as run:
            mlflow.set_tags({**COMMON_TAGS, "model_type": "gaussian_hmm",
                             "n_states": "3"})
            mlflow.log_params({
                "n_components"   : 3,
                "covariance_type": "diag",
                "n_iter"         : 150,
                "hmm_features"   : str(HMM_FEATURES),
                "n_sequences"    : len(sequences),
                "n_observations" : total_obs,
            })
            for s in range(3):
                lbl = STATE_NAMES[s].replace(" ", "_")
                for fi, fn in enumerate(HMM_FEATURES):
                    mlflow.log_metric(f"{lbl}_mean_{fn}",
                                      round(float(model_hmm.means_[s][fi]), 4))
                mlflow.log_metric(f"{lbl}_dwell_days",
                                  round(dwell_times[s], 2))
                mlflow.log_metric(f"{lbl}_prevalence_pct",
                                  round(100 * state_counts[s] / max(total_obs, 1), 2))

            trans_mat_df.to_csv("/tmp/hmm_transmat.csv")
            mlflow.log_artifact("/tmp/hmm_transmat.csv")
            joblib.dump(model_hmm, "/tmp/ps2_hmm.joblib")
            mlflow.log_artifact("/tmp/ps2_hmm.joblib", "hmm_model")
            HMM_RUN = run.info.run_id

        # ── Detailed printed summary ──────────────────────────────────────────
        print("\n" + "="*72)
        print("HMM — 3-State Gaussian Cascade Severity Regimes")
        print("="*72)
        print(f"  Sequences (devices with >=3 cascade days) : {len(sequences):,}")
        print(f"  Total observations (device-cascade-days)  : {total_obs:,}")
        print(f"  Features                                   : {HMM_FEATURES}")

        print(f"\n── State Means & Standard Deviations ──────────────────────────────────")
        for s in range(3):
            label = STATE_NAMES[s]
            means = model_hmm.means_[s]
            cov_s = model_hmm.covars_[s]
            stds  = np.sqrt(np.diag(cov_s) if cov_s.ndim == 2 else cov_s)
            prev  = 100 * state_counts[s] / max(total_obs, 1)
            dwell = dwell_times[s]
            print(f"\n  State {s} — {label}  "
                  f"({prev:.1f}% of cascade days  |  avg dwell {dwell:.1f} days)")
            print(f"  {'Feature':<30}  {'Mean':>9}  {'Std':>9}")
            print(f"  {'-'*50}")
            for fi, fn in enumerate(HMM_FEATURES):
                print(f"  {fn:<30}  {means[fi]:>9.3f}  {stds[fi]:>9.3f}")

        print(f"\n── Transition Matrix (P(next_state | current_state)) ───────────────────")
        print(f"  Rows = FROM state  |  Cols = TO state")
        print(trans_mat_df.to_string())

        print(f"\n── Dwell Times (expected consecutive days in each state) ───────────────")
        for s in range(3):
            p_self = model_hmm.transmat_[s, s]
            print(f"  State {s} ({STATE_NAMES[s]:<20}) "
                  f"P(stay)={p_self:.3f}  "
                  f"Expected dwell={dwell_times[s]:.1f} days")

        print(f"\n── Transition Interpretation ───────────────────────────────────────────")
        for i in range(3):
            exits = [(j, model_hmm.transmat_[i, j]) for j in range(3) if j != i]
            exits_sorted = sorted(exits, key=lambda x: -x[1])
            most_likely_next, prob_next = exits_sorted[0]
            print(f"  From {STATE_NAMES[i]:<20}: "
                  f"stay={model_hmm.transmat_[i,i]:.1%}  "
                  f"most likely exit → {STATE_NAMES[most_likely_next]} "
                  f"({prob_next:.1%})")

        print(f"\n── State Prevalence ────────────────────────────────────────────────────")
        for s in range(3):
            bar = "█" * int(30 * state_counts[s] / max(total_obs, 1))
            print(f"  State {s} ({STATE_NAMES[s]:<20}) "
                  f"{100*state_counts[s]/max(total_obs,1):5.1f}%  {bar}")

        # ── Emission distribution plots ───────────────────────────────────────
        n_feat = len(HMM_FEATURES)
        fig, axes = plt.subplots(1, n_feat, figsize=(4 * n_feat, 5))
        if n_feat == 1:
            axes = [axes]
        colors = ["#4CAF50", "#FF9800", "#F44336"]   # green/orange/red

        for fi, (fn, ax) in enumerate(zip(HMM_FEATURES, axes)):
            for s in range(3):
                mu  = model_hmm.means_[s][fi]
                cov2 = model_hmm.covars_[s]
                std = np.sqrt(cov2[fi, fi] if cov2.ndim == 2 else cov2[fi])
                x   = np.linspace(max(0, mu - 3*std), mu + 3*std, 200)
                y   = (1/(std * np.sqrt(2*np.pi))
                       * np.exp(-0.5 * ((x - mu)/std)**2))
                ax.plot(x, y, color=colors[s], linewidth=2,
                        label=f"S{s}: {STATE_NAMES[s][:12]}\n"
                              f"μ={mu:.1f}  σ={std:.1f}")
                ax.axvline(mu, color=colors[s], linestyle="--", alpha=0.4)
            ax.set_title(fn.replace("_", " "), fontsize=9)
            ax.set_xlabel("Value"); ax.set_ylabel("Density")
            ax.legend(fontsize=7)

        plt.suptitle("HMM State Emission Distributions per Feature", fontsize=11)
        plt.tight_layout()
        save_and_log(fig, "/tmp/ps2_hmm_emissions.png", run)

        # ── Transition matrix heatmap ─────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(6, 5))
        short_labels = [f"S{i}\n{STATE_NAMES[i][:10]}" for i in range(3)]
        im = ax.imshow(model_hmm.transmat_, cmap="YlOrRd", vmin=0, vmax=1)
        plt.colorbar(im, ax=ax, label="Transition probability")
        ax.set_xticks(range(3)); ax.set_yticks(range(3))
        ax.set_xticklabels([f"→ {l}" for l in short_labels])
        ax.set_yticklabels([f"From {l}" for l in short_labels])
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f"{model_hmm.transmat_[i,j]:.3f}",
                        ha="center", va="center", fontsize=11,
                        color="black" if model_hmm.transmat_[i,j] < 0.7 else "white",
                        fontweight="bold")
        ax.set_title("HMM Transition Matrix\n(row=current state, col=next state)",
                     fontsize=10)
        plt.tight_layout()
        save_and_log(fig, "/tmp/ps2_hmm_transmat.png", run)

        print(f"\nHMM analysis logged to MLflow (run={HMM_RUN})")

In [ ]:
mlflow.end_run()  # close any run left open by a prior cell's failure
# ── CELL 15 : Association Rules — co-failing subsystem pairs ──────────────────
# subsystem_chain separator = '->'
# Adaptive min_support per category — smaller subsets use lower threshold.

def run_association_rules(df_sub, label="overall", min_support=0.05,
                          min_lift=1.5, top_n=10):
    if "subsystem_chain" not in df_sub.columns:
        return None, 0, 0
    basket = [
        list(set(s.strip() for s in row.split("->") if s.strip()))
        for row in df_sub["subsystem_chain"].dropna()
    ]
    basket = [b for b in basket if len(b) >= 2]
    if not basket:
        print(f"  [{label}] No multi-item baskets — skipped")
        return None, 0, 0

    te     = TransactionEncoder()
    te_arr = te.fit(basket).transform(basket)
    bdf    = pd.DataFrame(te_arr, columns=te.columns_)

    try:
        freq = apriori(bdf, min_support=min_support, use_colnames=True)
        if freq.empty:
            # Try halving support once before giving up
            freq = apriori(bdf, min_support=min_support / 2, use_colnames=True)
            if freq.empty:
                print(f"  [{label}] No frequent itemsets even at "
                      f"support={min_support/2:.4f} (n_baskets={len(basket):,})")
                return None, len(basket), 0
            print(f"  [{label}] Lowered support to {min_support/2:.4f} "
                  f"({len(freq)} itemsets found)")

        # Progressive lift fallback: try original → 1.1 → 1.05 → 1.0 → conf only
        rules = pd.DataFrame()
        used_lift = None
        for try_lift in [min_lift, 1.1, 1.05, 1.0]:
            candidate = association_rules(freq, metric="lift",
                                          min_threshold=try_lift)
            if not candidate.empty:
                rules = candidate
                used_lift = try_lift
                if try_lift < min_lift:
                    print(f"  [{label}] Lowered lift threshold to {try_lift:.2f} "
                          f"(original {min_lift} yielded 0 rules)")
                break

        if rules.empty:
            # Final fallback: show top 2-item frequent sets by support
            pairs = freq[freq["itemsets"].apply(len) == 2].copy()
            singles = freq[freq["itemsets"].apply(len) == 1].copy()
            print(f"  [{label}] No rules found (lift stays near 1.0 — "
                  f"subsystems co-occur at ~random frequency)")
            if not pairs.empty:
                pairs = pairs.sort_values("support", ascending=False).head(10)
                pairs["pair"] = pairs["itemsets"].apply(
                    lambda x: " + ".join(sorted(x)))
                print(f"  Top co-occurring pairs by support "
                      f"(lift~1.0 = independent but frequent):")
                print(pairs[["pair","support"]].to_string(index=False))
            if not singles.empty:
                singles = singles.sort_values("support", ascending=False).head(5)
                singles["subsystem"] = singles["itemsets"].apply(
                    lambda x: ", ".join(sorted(x)))
                print(f"  Top single subsystems in multi-item chains:")
                print(singles[["subsystem","support"]].to_string(index=False))
            return None, len(basket), 0

        rules = rules.sort_values("lift", ascending=False)
        out   = rules[["antecedents", "consequents", "support",
                        "confidence", "lift", "leverage", "conviction"]].copy()
        out["antecedents"] = out["antecedents"].apply(lambda x: ", ".join(sorted(x)))
        out["consequents"] = out["consequents"].apply(lambda x: ", ".join(sorted(x)))
        out["conviction"]  = out["conviction"].clip(upper=99.0)
        if used_lift is not None and used_lift < min_lift:
            out["_note"] = f"lift_threshold={used_lift}"
        return out, len(basket), len(rules)
    except Exception as e:
        print(f"  [{label}] Association rules error: {e}")
        return None, len(basket), 0

if "subsystem_chain" not in df.columns:
    print("subsystem_chain not available — association rules skipped")
    RULES_RUN = None
else:
    with mlflow.start_run(run_name="association_rules") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "association_rules"})
        GLOBAL_MIN_SUPPORT = 0.05
        GLOBAL_MIN_LIFT    = 1.5
        mlflow.log_params({"min_support_global": GLOBAL_MIN_SUPPORT,
                           "min_lift": GLOBAL_MIN_LIFT})

        # ── 1. Overall rules ──────────────────────────────────────────────────
        rules_all, n_baskets_all, n_rules_all = run_association_rules(
            train_df, label="overall",
            min_support=GLOBAL_MIN_SUPPORT, min_lift=GLOBAL_MIN_LIFT, top_n=10)
        mlflow.log_param("n_baskets_overall", n_baskets_all)
        mlflow.log_param("n_rules_overall",   n_rules_all)

        if rules_all is not None:
            rules_all.to_csv("/tmp/association_rules_all.csv", index=False)
            mlflow.log_artifact("/tmp/association_rules_all.csv")
            for i, row in rules_all.head(10).iterrows():
                mlflow.log_metric(f"rule_overall_{i}_lift",
                                  round(float(row["lift"]), 3))

            pd.set_option("display.max_colwidth", 35)
            print(f"\n── Top 10 Association Rules (overall) ─────────────────────────────────")
            print(f"   n_baskets={n_baskets_all:,}  |  total rules found={n_rules_all}")
            print(rules_all.head(10).to_string(index=False))

            print(f"\n── Rule strength summary ───────────────────────────────────────────────")
            print(f"  Rules lift > 2  : {(rules_all['lift'] > 2).sum():>5}")
            print(f"  Rules lift > 5  : {(rules_all['lift'] > 5).sum():>5}")
            print(f"  Rules lift > 10 : {(rules_all['lift'] > 10).sum():>5}")
            print(f"  Max lift        : {rules_all['lift'].max():.3f}")
            print(f"  Max confidence  : {rules_all['confidence'].max():.3f}")
            print(f"  Max conviction  : {rules_all['conviction'].max():.3f}")

            # ── Visualisation: bubble chart + lift-ranked bar ─────────────────
            plot_df = rules_all.head(20).copy()
            plot_df["rule_label"] = (plot_df["antecedents"].str[:20]
                                     + " → " + plot_df["consequents"].str[:20])
            fig, axes = plt.subplots(1, 2, figsize=(16, 6))

            sc = axes[0].scatter(
                plot_df["support"], plot_df["confidence"],
                s=plot_df["lift"] * 25, c=plot_df["lift"],
                cmap="YlOrRd", alpha=0.85, edgecolors="black", linewidth=0.5)
            plt.colorbar(sc, ax=axes[0], label="Lift")
            axes[0].set_xlabel("Support"); axes[0].set_ylabel("Confidence")
            axes[0].set_title("Top-20 Rules: Support vs Confidence\n"
                              "(bubble size/color = Lift)")
            for _, r in plot_df.iterrows():
                axes[0].annotate(r["rule_label"][:28],
                                 (r["support"], r["confidence"]),
                                 fontsize=6, ha="center", va="bottom", alpha=0.7)

            max_lift = plot_df["lift"].max() or 1
            bar_colors = plt.cm.YlOrRd(plot_df["lift"][::-1].values / max_lift)
            axes[1].barh(range(len(plot_df)),
                         plot_df["lift"][::-1].values, color=bar_colors)
            axes[1].set_yticks(range(len(plot_df)))
            axes[1].set_yticklabels(plot_df["rule_label"][::-1].tolist(), fontsize=7)
            axes[1].set_xlabel("Lift")
            axes[1].set_title("Top-20 Rules ranked by Lift")
            axes[1].axvline(1.0, color="gray", linestyle="--",
                            alpha=0.5, label="lift=1 (random)")
            axes[1].legend(fontsize=7)

            plt.suptitle("Association Rules — Co-Failing Subsystem Patterns", fontsize=11)
            plt.tight_layout()
            save_and_log(fig, "/tmp/ps2_association_rules_chart.png", run)

        # ── 2. Per device category (adaptive min_support) ─────────────────────
        if "mars_device_category" in df.columns:
            print(f"\n{'='*70}")
            print("Per-Category Association Rules  (adaptive min_support)")
            print(f"{'='*70}")
            for cat in sorted(train_df["mars_device_category"].dropna().unique()):
                df_cat     = train_df[train_df["mars_device_category"] == cat]
                n_cat_rows = len(df_cat)
                # Adaptive threshold — smaller subsets need lower support
                cat_support = (0.01 if n_cat_rows < 2_000  else
                               0.02 if n_cat_rows < 10_000 else
                               0.03 if n_cat_rows < 50_000 else 0.05)
                cat_lift = 1.0  # lowered: sparse categories rarely exceed 1.2

                # Print header BEFORE calling function so fallback output lands below it
                print(f"\n── [{cat}]  n_rows={n_cat_rows:,}  "
                      f"min_support={cat_support}  min_lift={cat_lift} ──────────")
                rules_cat, n_b, n_r = run_association_rules(
                    df_cat, label=cat,
                    min_support=cat_support, min_lift=cat_lift, top_n=5)
                mlflow.log_param(f"min_support_{cat}", cat_support)
                mlflow.log_param(f"n_baskets_{cat}",   n_b)
                mlflow.log_param(f"n_rules_{cat}",     n_r)
                print(f"   n_baskets={n_b:,}  rules_found={n_r}")

                if rules_cat is not None:
                    csv_path = f"/tmp/association_rules_{cat.lower()}.csv"
                    rules_cat.to_csv(csv_path, index=False)
                    mlflow.log_artifact(csv_path)
                    for i, row in rules_cat.head(5).iterrows():
                        mlflow.log_metric(f"rule_{cat}_{i}_lift",
                                          round(float(row["lift"]), 3))
                    print(rules_cat.head(5).to_string(index=False))
                elif n_b == 0:
                    # Truly no multi-item baskets at all
                    print("  No multi-item cascade chains — association rules not applicable.")
                # else: fallback pairs/singles already printed inside run_association_rules()

        RULES_RUN = run.info.run_id
    print(f"\nAssociation rules logged (run={RULES_RUN})")

In [ ]:
# ── CELL 16 : Network Graph — cascade relationship visualization ───────────────
# Directed graph: nodes = subsystems, edges = directional cascade transitions.
# Node SIZE  = betweenness centrality (cascade routing importance).
# Node COLOR = betweenness centrality (yellow→red gradient via YlOrRd).
# Edge WIDTH = transition count (thicker = more frequent).
# Diagram is rendered inline + logged to MLflow as PNG artifact.

def build_and_plot_network(df_sub, title, path, run=None, min_edge_weight=2):
    if "subsystem_chain" not in df_sub.columns:
        print(f"  subsystem_chain missing — network skipped ({title})")
        return None

    chains = (df_sub["subsystem_chain"]
              .dropna()
              .apply(lambda s: [x.strip() for x in str(s).split("->") if x.strip()])
              .tolist())
    chains = [c for c in chains if len(c) >= 2]
    if not chains:
        print(f"  No chains — network skipped ({title})")
        return None

    edge_counts = defaultdict(int)
    node_counts = Counter()
    for chain in chains:
        for s in chain:
            node_counts[s] += 1
        for a, b in zip(chain[:-1], chain[1:]):
            edge_counts[(a, b)] += 1

    G = nx.DiGraph()
    for node, cnt in node_counts.items():
        G.add_node(node, count=cnt)
    for (a, b), w in edge_counts.items():
        if w >= min_edge_weight:
            G.add_edge(a, b, weight=w)

    if G.number_of_edges() == 0:
        print(f"  No edges above min_weight={min_edge_weight} — skipped ({title})")
        return G

    # ── Centrality for visual encoding ────────────────────────────────────────
    between  = nx.betweenness_centrality(G, weight="weight")
    max_btw  = max(between.values()) if between else 1e-9
    try:
        pagerank = nx.pagerank(G, weight="weight", max_iter=300)
    except Exception:
        pagerank = {n: 1 / max(len(G), 1) for n in G.nodes()}

    # ── Layout (kamada-kawai for clarity on small graphs) ─────────────────────
    try:
        pos = (nx.kamada_kawai_layout(G)
               if G.number_of_nodes() <= 15
               else nx.spring_layout(G, seed=42, k=2.5))
    except Exception:
        pos = nx.circular_layout(G)

    # ── Visual encoding ───────────────────────────────────────────────────────
    node_sizes  = [600 + 4500 * between.get(n, 0) / max_btw for n in G.nodes()]
    node_colors = [between.get(n, 0) / max_btw for n in G.nodes()]
    edge_weights = [G[u][v]["weight"] for u, v in G.edges()]
    max_w       = max(edge_weights) if edge_weights else 1
    edge_widths = [0.8 + 5.0 * (w / max_w) for w in edge_weights]
    edge_alphas = [0.35 + 0.65 * (w / max_w) for w in edge_weights]
    # RGBA edge color: steel-blue with alpha mapped to frequency
    edge_rgba   = [(0.27, 0.51, 0.71, a) for a in edge_alphas]

    fig, ax = plt.subplots(figsize=(14, 10))
    cmap = plt.cm.YlOrRd
    norm = plt.Normalize(vmin=0, vmax=1)

    nx.draw_networkx_nodes(G, pos, ax=ax,
                           node_size=node_sizes,
                           node_color=node_colors,
                           cmap=cmap, vmin=0, vmax=1,
                           alpha=0.92, linewidths=1.0,
                           edgecolors="white")
    nx.draw_networkx_labels(G, pos, ax=ax,
                            font_size=8, font_color="black",
                            font_weight="bold")
    nx.draw_networkx_edges(G, pos, ax=ax,
                           width=edge_widths,
                           edge_color=edge_rgba,
                           arrows=True, arrowsize=16,
                           connectionstyle="arc3,rad=0.12",
                           min_source_margin=28, min_target_margin=28)

    # Edge weight labels for top-12 transitions
    top_edges = sorted(edge_counts.items(), key=lambda x: -x[1])[:12]
    edge_labels = {(a, b): f"{w:,}" for (a, b), w in top_edges
                   if G.has_edge(a, b)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax,
                                 font_size=7, label_pos=0.35,
                                 bbox=dict(boxstyle="round,pad=0.2",
                                           fc="white", alpha=0.7))

    plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm),
                 ax=ax, label="Betweenness Centrality (relative)",
                 shrink=0.55, pad=0.02)

    # Node-size legend
    for rel, lbl in [(0.05, "Low hub"), (0.4, "Med hub"), (0.9, "High hub")]:
        ax.scatter([], [], s=600 + 4500 * rel, c="gray", alpha=0.6,
                   label=f"{lbl}")
    ax.legend(scatterpoints=1, frameon=True, labelspacing=1.1,
              loc="lower left", fontsize=8, title="Node size = hub score")

    ax.set_title(title, fontsize=11, pad=14)
    ax.axis("off")
    plt.tight_layout()
    save_and_log(fig, path, run)   # save_and_log calls plt.show() + plt.close()

    return G

# ── Main ──────────────────────────────────────────────────────────────────────
if "subsystem_chain" not in df.columns:
    print("subsystem_chain not available — network graph skipped")
    NETWORK_RUN = None
else:
    with mlflow.start_run(run_name="cascade_network_graph") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "networkx_cascade_graph"})
        mlflow.log_param("min_edge_weight", 2)

        # ── Overall network ───────────────────────────────────────────────────
        G_all = build_and_plot_network(
            df,
            "PS2 — Cascade Subsystem Network (all devices)\n"
            "Node color/size = betweenness centrality  |  Edge width = transition count",
            "/tmp/ps2_network_all.png", run=run)

        if G_all is not None and G_all.number_of_nodes() > 0:
            mlflow.log_metric("network_n_nodes", G_all.number_of_nodes())
            mlflow.log_metric("network_n_edges", G_all.number_of_edges())
            mlflow.log_metric("network_density", round(nx.density(G_all), 4))

        if G_all is not None and G_all.number_of_nodes() > 1:
            # ── Full centrality + PageRank table ──────────────────────────────
            in_cent   = nx.in_degree_centrality(G_all)
            out_cent  = nx.out_degree_centrality(G_all)
            between   = nx.betweenness_centrality(G_all, weight="weight")
            try:
                pagerank  = nx.pagerank(G_all, weight="weight", max_iter=300)
                closeness = nx.closeness_centrality(G_all)
            except Exception:
                pagerank  = {n: 1/len(G_all) for n in G_all.nodes()}
                closeness = {n: 0.0 for n in G_all.nodes()}

            centrality_df = pd.DataFrame({
                "in_degree"  : in_cent,
                "out_degree" : out_cent,
                "betweenness": between,
                "pagerank"   : pagerank,
                "closeness"  : closeness,
            }).sort_values("betweenness", ascending=False)
            centrality_df.to_csv("/tmp/ps2_network_centrality.csv")
            mlflow.log_artifact("/tmp/ps2_network_centrality.csv")

            pd.set_option("display.float_format", "{:.4f}".format)
            print("\n── Cascade Subsystem Hub Analysis (all devices) ────────────────────────")
            print(f"   Nodes: {G_all.number_of_nodes()}  |  "
                  f"Edges: {G_all.number_of_edges()}  |  "
                  f"Density: {nx.density(G_all):.4f}")
            print("\n  Subsystems ranked by betweenness centrality:")
            print(centrality_df.to_string())

            print("\n── Node role classification ────────────────────────────────────────────")
            for node in centrality_df.index:
                in_d  = G_all.in_degree(node)
                out_d = G_all.out_degree(node)
                btw   = between[node]
                role  = ("Source"     if in_d == 0 and out_d > 0 else
                         "Sink"       if out_d == 0 and in_d > 0 else
                         "Major Hub"  if btw > 0.3 else
                         "Minor Hub"  if btw > 0.1 else
                         "Relay"      if btw > 0.01 else
                         "Peripheral")
                print(f"  {node:<22}  in={in_d:>2}  out={out_d:>2}  "
                      f"btw={btw:.3f}  pr={pagerank[node]:.4f}  → {role}")

            print("\n── Interpretation ──────────────────────────────────────────────────────")
            top_btw = max(between, key=between.get)
            top_pr  = max(pagerank, key=pagerank.get)
            print(f"  Top cascade router (betweenness) : {top_btw}")
            print(f"    Lies on most cascade paths — fixing {top_btw} has highest cascade-break impact")
            print(f"  Most cascade-visited (PageRank)  : {top_pr}")
            print(f"    Receives influence from broadest set of upstream subsystems")

            # Log top-5 by betweenness + pagerank
            for node, v in sorted(between.items(), key=lambda x: -x[1])[:5]:
                mlflow.log_metric(f"centrality_btw_{node}", round(v, 4))
            for node, v in sorted(pagerank.items(), key=lambda x: -x[1])[:5]:
                mlflow.log_metric(f"pagerank_{node}", round(v, 4))

            # ── Top 3-step cascade paths ──────────────────────────────────────
            all_chains_net = (df["subsystem_chain"].dropna()
                              .apply(lambda s: [x.strip()
                                                for x in str(s).split("->")
                                                if x.strip()])
                              .tolist())
            path3_counts = Counter()
            for c in all_chains_net:
                for i in range(len(c) - 2):
                    path3_counts[tuple(c[i:i+3])] += 1
            print("\n── Top-10 Most Common 3-Step Cascade Paths ─────────────────────────────")
            for path_t, cnt in path3_counts.most_common(10):
                print(f"  {' → '.join(path_t):<55}  count={cnt:>8,}")

        # ── Per device category ───────────────────────────────────────────────
        if "mars_device_category" in df.columns:
            print(f"\n{'='*70}")
            print("Per-Category Network Graphs")
            print(f"{'='*70}")
            for cat in sorted(df["mars_device_category"].dropna().unique()):
                df_cat = df[df["mars_device_category"] == cat]
                G_cat  = build_and_plot_network(
                    df_cat,
                    f"PS2 — Cascade Subsystem Network [{cat}]\n"
                    f"Node color/size = betweenness  |  Edge width = count",
                    f"/tmp/ps2_network_{cat.lower()}.png",
                    run=run)
                if G_cat is not None:
                    mlflow.log_metric(f"network_{cat}_n_nodes",
                                      G_cat.number_of_nodes())
                    mlflow.log_metric(f"network_{cat}_n_edges",
                                      G_cat.number_of_edges())

        NETWORK_RUN = run.info.run_id
    print(f"\nNetwork graph analysis logged (run={NETWORK_RUN})")

In [ ]:
# ── CELL 17 : Cascade Ignition & Termination Analysis ────────────────────────
# first_subsystem = subsystem that STARTS the cascade (where to intervene)
# last_subsystem  = subsystem where cascade ENDS (what finally fails)

if "first_subsystem" not in df.columns or "last_subsystem" not in df.columns:
    print("first_subsystem / last_subsystem not in Gold — ignition analysis skipped")
    IGNITION_RUN = None
else:
    with mlflow.start_run(run_name="cascade_ignition_analysis") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "ignition_termination_analysis"})

        # ── 1. Overall ignition distribution ─────────────────────────────────
        first_counts = df["first_subsystem"].value_counts()
        last_counts  = df["last_subsystem"].value_counts()

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        first_counts.head(15).plot(kind="bar", ax=axes[0], color="#E53935", edgecolor="white")
        axes[0].set_title("Top 15 Cascade Ignition Subsystems (first fault)")
        axes[0].set_xlabel(""); axes[0].set_ylabel("Cascade days")
        axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha="right")
        last_counts.head(15).plot(kind="bar", ax=axes[1], color="#1565C0", edgecolor="white")
        axes[1].set_title("Top 15 Cascade Termination Subsystems (last fault)")
        axes[1].set_xlabel(""); axes[1].set_ylabel("Cascade days")
        axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha="right")
        plt.suptitle("Cascade Ignition vs Termination — Where do cascades START and END?",
                     fontsize=11)
        plt.tight_layout()
        save_and_log(fig, "/tmp/ps2_ignition_termination.png", run)

        # ── 2. First → Last cross-heatmap ─────────────────────────────────────
        top_first = first_counts.head(10).index.tolist()
        top_last  = last_counts.head(10).index.tolist()
        fl_cross  = (df[df["first_subsystem"].isin(top_first) &
                        df["last_subsystem"].isin(top_last)]
                     .groupby(["first_subsystem", "last_subsystem"])
                     .size().unstack("last_subsystem", fill_value=0)
                     .reindex(index=top_first, columns=top_last, fill_value=0))
        fig, ax = plt.subplots(figsize=(12, 8))
        sns.heatmap(fl_cross, annot=True, fmt="d", cmap="YlOrRd",
                    ax=ax, linewidths=0.4, annot_kws={"size": 8})
        ax.set_title("Cascade Flow: First Subsystem → Last Subsystem\n"
                     "(top-10 each, count = cascade days)", fontsize=11)
        ax.set_xlabel("Last Subsystem (where cascade ends)")
        ax.set_ylabel("First Subsystem (where cascade starts)")
        ax.tick_params(axis="x", rotation=45, labelsize=8)
        ax.tick_params(axis="y", rotation=0,  labelsize=8)
        plt.tight_layout()
        save_and_log(fig, "/tmp/ps2_first_last_heatmap.png", run)
        fl_cross.to_csv("/tmp/ps2_first_last_cross.csv")
        mlflow.log_artifact("/tmp/ps2_first_last_cross.csv")

        # ── 3. Print summaries ────────────────────────────────────────────────
        print("\n── Cascade Ignition Analysis ──────────────────────────────────────────")
        print(f"\n  Top 15 cascade ignition subsystems (overall):")
        print(first_counts.head(15).to_string())
        print(f"\n  Top 15 cascade termination subsystems (overall):")
        print(last_counts.head(15).to_string())

        if "mars_device_category" in df.columns:
            print(f"\n── Per-Category Cascade Ignition ──────────────────────────────────────")
            for cat in sorted(df["mars_device_category"].dropna().unique()):
                df_cat = df[df["mars_device_category"] == cat]
                fc_cat = df_cat["first_subsystem"].value_counts().head(5)
                lc_cat = df_cat["last_subsystem"].value_counts().head(5)
                print(f"\n  [{cat}] ({len(df_cat):,} cascade days)")
                print(f"    Top 5 ignition : {dict(fc_cat)}")
                print(f"    Top 5 terminus : {dict(lc_cat)}")

        # ── 4. Self-cascades (same subsystem start and end) ───────────────────
        self_loops = df[df["first_subsystem"] == df["last_subsystem"]]
        print(f"\n── Self-Cascades (first == last subsystem) ────────────────────────────")
        print(f"  Count : {len(self_loops):,}  "
              f"({100*len(self_loops)/max(len(df),1):.1f}% of all cascade days)")
        print(f"  Top 5 self-cascading subsystems:")
        print(self_loops["first_subsystem"].value_counts().head(5).to_string())

        # ── 5. Top device per top-5 ignition subsystem ───────────────────────
        print(f"\n── Top Device per Ignition Subsystem (top-5 ignitors) ─────────────────")
        for sub in first_counts.head(5).index:
            df_sub   = df[df["first_subsystem"] == sub]
            top_dev  = df_sub.groupby("DEVICE_ID").size().idxmax()
            top_cnt  = df_sub.groupby("DEVICE_ID").size().max()
            print(f"  {sub:<20}: {len(df_sub):>8,} cascade days | "
                  f"worst device = {top_dev} ({top_cnt:,} days)")

        for sub, cnt in first_counts.head(10).items():
            mlflow.log_metric(f"ignition_{sub}", int(cnt))
        for sub, cnt in last_counts.head(10).items():
            mlflow.log_metric(f"terminus_{sub}", int(cnt))

        IGNITION_RUN = run.info.run_id
    print(f"\nIgnition analysis logged (run={IGNITION_RUN})")


In [ ]:
# ── CELL 18 : Device Recurrence — Chronic Cascade Offender Profiling ─────────
# Which devices cascade week-after-week (chronic) vs sporadically?
# Rolling monthly trend for top devices reveals deterioration patterns.

with mlflow.start_run(run_name="device_recurrence_analysis") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "device_recurrence"})

    # ── 1. Per-device statistics ──────────────────────────────────────────────
    dev_stats = df.groupby("DEVICE_ID").agg(
        cascade_days  = ("transit_day", "count"),
        first_cascade = ("transit_day", "min"),
        last_cascade  = ("transit_day", "max"),
        avg_chain_len = ("chain_length", "mean"),
        max_chain_len = ("chain_length", "max"),
    )
    dev_stats["active_days"]  = (dev_stats["last_cascade"] - dev_stats["first_cascade"]).dt.days + 1
    dev_stats["cascade_rate"] = dev_stats["cascade_days"] / dev_stats["active_days"].clip(lower=1)
    if "mars_device_category" in df.columns:
        dc = df.drop_duplicates("DEVICE_ID").set_index("DEVICE_ID")["mars_device_category"]
        dev_stats["category"] = dev_stats.index.map(dc)
    dev_stats = dev_stats.sort_values("cascade_days", ascending=False)
    dev_stats.to_csv("/tmp/ps2_device_recurrence.csv")
    mlflow.log_artifact("/tmp/ps2_device_recurrence.csv")

    print("\n── Device Recurrence Summary ───────────────────────────────────────────")
    print(f"  Total devices        : {len(dev_stats):,}")
    print(f"  Cascades once only   : {(dev_stats['cascade_days']==1).sum():,}")
    print(f"  Cascades 10+ times   : {(dev_stats['cascade_days']>=10).sum():,}")
    print(f"  Cascades 100+ times  : {(dev_stats['cascade_days']>=100).sum():,}")
    print(f"\n── Top 20 Chronic Offenders (most cascade days) ────────────────────────")
    print(dev_stats.head(20).to_string())

    # ── 2. Distribution plots ─────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    dev_stats["cascade_days"].clip(upper=500).plot(
        kind="hist", bins=40, ax=axes[0], color="#E53935", edgecolor="white")
    axes[0].set_title("Total cascade days per device (clipped @500)")
    axes[0].set_xlabel("Cascade days")

    dev_stats["cascade_rate"].clip(upper=0.5).plot(
        kind="hist", bins=40, ax=axes[1], color="#7B1FA2", edgecolor="white")
    axes[1].set_title("Daily cascade rate (cascade_days / active_days)")
    axes[1].set_xlabel("Cascade rate (clipped @0.5)")

    dev_stats["avg_chain_len"].clip(upper=30).plot(
        kind="hist", bins=30, ax=axes[2], color="#1565C0", edgecolor="white")
    axes[2].set_title("Average chain length per device")
    axes[2].set_xlabel("Mean chain length")

    plt.suptitle("Device Recurrence — Chronic vs Sporadic Cascade Patterns", fontsize=11)
    plt.tight_layout()
    save_and_log(fig, "/tmp/ps2_device_recurrence_dist.png", run)

    # ── 3. Monthly cascade trend for top 5 devices ────────────────────────────
    top5_ids = dev_stats.index[:5].tolist()
    fig, ax  = plt.subplots(figsize=(15, 6))
    colors   = ["#E53935", "#1565C0", "#2E7D32", "#F57F17", "#6A1B9A"]
    for i, dev_id in enumerate(top5_ids):
        df_dev  = df[df["DEVICE_ID"] == dev_id].set_index("transit_day")
        monthly = df_dev.resample("ME").size().rename(dev_id)
        n_days  = dev_stats.loc[dev_id, "cascade_days"]
        monthly.plot(ax=ax, color=colors[i], marker="o", linewidth=1.5, markersize=4,
                     label=f"{dev_id} ({n_days} days)")
    ax.set_title("Monthly Cascade Day Count — Top 5 Chronic Devices")
    ax.set_xlabel("Month"); ax.set_ylabel("Cascade days")
    ax.legend(fontsize=8, title="Device (total cascade days)")
    ax.axhline(0, color="gray", linewidth=0.5)
    plt.tight_layout()
    save_and_log(fig, "/tmp/ps2_top5_device_monthly.png", run)

    # ── 4. Chronic vs sporadic classification ─────────────────────────────────
    chronic_thresh = dev_stats["cascade_days"].quantile(0.90)
    dev_stats["recurrence_class"] = "sporadic"
    dev_stats.loc[dev_stats["cascade_days"] >= chronic_thresh, "recurrence_class"] = "chronic"
    rc = dev_stats["recurrence_class"].value_counts()
    print(f"\n── Recurrence Classification (90th pct = {chronic_thresh:.0f} cascade days) ──────")
    print(f"  Chronic  (>={chronic_thresh:.0f} days) : {rc.get('chronic',0):>5,} devices")
    print(f"  Sporadic (<{chronic_thresh:.0f} days)  : {rc.get('sporadic',0):>5,} devices")
    if "category" in dev_stats.columns:
        print(f"\n  Chronic devices by category:")
        print(dev_stats[dev_stats["recurrence_class"]=="chronic"]["category"]
              .value_counts().to_string())

    # ── 5. Top device weekly cascade timeline ─────────────────────────────────
    top_dev    = dev_stats.index[0]
    df_top     = df[df["DEVICE_ID"] == top_dev].set_index("transit_day")
    weekly_top = df_top.resample("W").size()
    print(f"\n── Top Chronic Device: {top_dev} ───────────────────────────────────────")
    print(f"  Cascade days       : {dev_stats.loc[top_dev,'cascade_days']:,}")
    print(f"  Daily cascade rate : {dev_stats.loc[top_dev,'cascade_rate']:.3f}")
    print(f"  Active span        : {dev_stats.loc[top_dev,'active_days']:,} days")
    print(f"  Avg chain length   : {dev_stats.loc[top_dev,'avg_chain_len']:.2f}")
    print(f"\n  Weekly cascade days (last 12 weeks):")
    for dt, n in weekly_top.tail(12).items():
        bar = "█" * n
        print(f"    {dt.strftime('%Y-%W')}: {n:>4}  {bar}")

    for metric, val in [
        ("top_device_cascade_rate",    dev_stats.iloc[0]["cascade_rate"]),
        ("n_chronic_devices",          int(rc.get("chronic", 0))),
        ("n_devices_100plus_cascades", int((dev_stats["cascade_days"] >= 100).sum())),
    ]:
        mlflow.log_metric(metric, round(float(val), 4))

    RECURRENCE_RUN = run.info.run_id
print(f"\nDevice recurrence analysis logged (run={RECURRENCE_RUN})")


In [ ]:
# ── CELL 19 : Business Impact Scoring ─────────────────────────────────────────
# Impact score = chain_length × type_weight  (floor = 1)
#   has_cash_cascade=3, has_gate_mech=2, has_printer=2, has_csc_reader=1
# Ranks devices and cascade ignitors by accumulated operational severity.

IMPACT_WEIGHTS = {
    "has_cash_cascade"        : 3,
    "has_gate_mech_in_chain"  : 2,
    "has_printer_in_chain"    : 2,
    "has_csc_reader_in_chain" : 1,
}
present_weights = {k: v for k, v in IMPACT_WEIGHTS.items() if k in df.columns}

if not present_weights:
    print("No impact flag columns in Gold — business impact analysis skipped")
    IMPACT_RUN = None
else:
    df_imp = df.copy()
    type_score = sum(df_imp[c].fillna(0) * w for c, w in present_weights.items()).clip(lower=1)
    df_imp["impact_score"] = df_imp["chain_length"] * type_score

    with mlflow.start_run(run_name="business_impact_scoring") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "business_impact_scoring"})
        mlflow.log_params({f"weight_{k}": v for k, v in present_weights.items()})

        # ── 1. Impact score distribution ──────────────────────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        clip99 = df_imp["impact_score"].quantile(0.99)
        df_imp["impact_score"].clip(upper=clip99).plot(
            kind="hist", bins=40, ax=axes[0], color="#E53935", edgecolor="white")
        axes[0].set_title("Business Impact Score Distribution (clipped @99th pct)")
        axes[0].set_xlabel("Impact score (chain_length × type weight)")
        if "mars_device_category" in df_imp.columns:
            (df_imp.groupby("mars_device_category")["impact_score"].mean()
             .plot(kind="bar", ax=axes[1], color="#7B1FA2", edgecolor="white"))
            axes[1].set_title("Mean Business Impact Score by Device Category")
            axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
        plt.suptitle("Business Impact — Operational Severity per Cascade Day", fontsize=11)
        plt.tight_layout()
        save_and_log(fig, "/tmp/ps2_impact_distribution.png", run)

        # ── 2. Top 20 highest-impact devices (total accumulated) ──────────────
        dev_impact = (df_imp.groupby("DEVICE_ID")["impact_score"].agg(
                          total_impact="sum", cascade_days="count",
                          avg_impact="mean", max_impact="max")
                      .sort_values("total_impact", ascending=False))
        if "mars_device_category" in df_imp.columns:
            dc = df_imp.drop_duplicates("DEVICE_ID").set_index("DEVICE_ID")["mars_device_category"]
            dev_impact["category"] = dev_impact.index.map(dc)
        dev_impact.to_csv("/tmp/ps2_device_impact.csv")
        mlflow.log_artifact("/tmp/ps2_device_impact.csv")
        print("\n── Top 20 Highest Business-Impact Devices (total accumulated score) ────")
        print(dev_impact.head(20).to_string())

        # ── 3. Top 20 by average impact per day (>= 5 cascade days) ──────────
        dev_avg = (dev_impact[dev_impact["cascade_days"] >= 5]
                   .sort_values("avg_impact", ascending=False))
        print(f"\n── Top 20 Devices by Average Impact Per Day (>=5 cascade days) ─────────")
        print(dev_avg.head(20).to_string())

        # ── 4. Impact by cascade ignition subsystem ───────────────────────────
        if "first_subsystem" in df_imp.columns:
            sub_impact = (df_imp.groupby("first_subsystem")["impact_score"].agg(
                              total_impact="sum", avg_impact="mean", count="count")
                          .sort_values("total_impact", ascending=False))
            sub_impact.to_csv("/tmp/ps2_subsystem_impact.csv")
            mlflow.log_artifact("/tmp/ps2_subsystem_impact.csv")
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            sub_impact.head(15)["total_impact"].plot(
                kind="barh", ax=axes[0], color="#E53935", edgecolor="white")
            axes[0].set_title("Total Business Impact by Cascade Ignitor")
            axes[0].set_xlabel("Accumulated impact score"); axes[0].invert_yaxis()
            sub_impact.head(15)["avg_impact"].plot(
                kind="barh", ax=axes[1], color="#F57F17", edgecolor="white")
            axes[1].set_title("Average Impact per Cascade Day by Ignitor")
            axes[1].set_xlabel("Mean impact score per day"); axes[1].invert_yaxis()
            plt.suptitle("Business Impact by Cascade-Starting Subsystem", fontsize=11)
            plt.tight_layout()
            save_and_log(fig, "/tmp/ps2_subsystem_impact.png", run)
            print(f"\n── Business Impact by Cascade Ignition Subsystem ──────────────────────")
            print(sub_impact.head(15).to_string())

        # ── 5. Impact by time window ───────────────────────────────────────────
        if "window_bucket" in df_imp.columns:
            win_impact = (df_imp.groupby("window_bucket", observed=True)["impact_score"]
                          .agg(total_impact="sum", avg_impact="mean", count="count")
                          .reindex(list(TIME_WINDOWS.keys())))
            print(f"\n── Business Impact by Time Window ───────────────────────────────────────")
            print(win_impact.to_string())

        # ── 6. High-impact cascade composition (top 5%) ───────────────────────
        p95 = df_imp["impact_score"].quantile(0.95)
        high_impact = df_imp[df_imp["impact_score"] >= p95]
        print(f"\n── High-Impact Cascades (top 5%, score >= {p95:.0f}) ─────────────────────")
        print(f"  Count : {len(high_impact):,} cascade days")
        if "mars_device_category" in high_impact.columns:
            print(f"  By category  :\n"
                  f"{high_impact['mars_device_category'].value_counts().to_string()}")
        if "first_subsystem" in high_impact.columns:
            print(f"  Top ignitors :\n"
                  f"{high_impact['first_subsystem'].value_counts().head(8).to_string()}")

        # ── 7. Monthly impact trend for top 5 devices ─────────────────────────
        top5_imp = dev_impact.index[:5].tolist()
        fig, ax  = plt.subplots(figsize=(15, 6))
        colors   = ["#E53935", "#1565C0", "#2E7D32", "#F57F17", "#6A1B9A"]
        for i, did in enumerate(top5_imp):
            ts = (df_imp[df_imp["DEVICE_ID"] == did]
                  .set_index("transit_day")["impact_score"].resample("ME").sum())
            total = dev_impact.loc[did, "total_impact"]
            ts.plot(ax=ax, color=colors[i], linewidth=1.5, marker="o", markersize=4,
                    label=f"{did} (total={total:.0f})")
        ax.set_title("Monthly Accumulated Business Impact — Top 5 Highest-Impact Devices")
        ax.set_xlabel("Month"); ax.set_ylabel("Monthly total impact score")
        ax.legend(fontsize=8); plt.tight_layout()
        save_and_log(fig, "/tmp/ps2_top5_device_impact_timeline.png", run)

        for metric, val in [
            ("total_impact_all",           df_imp["impact_score"].sum()),
            ("avg_impact_per_cascade_day", df_imp["impact_score"].mean()),
            ("p95_impact_threshold",       p95),
            ("n_high_impact_days",         len(high_impact)),
        ]:
            mlflow.log_metric(metric, round(float(val), 2))

        IMPACT_RUN = run.info.run_id
    print(f"\nBusiness impact analysis logged (run={IMPACT_RUN})")


In [ ]:
# ── CELL 20 : Facility-Level Cascade Contagion ────────────────────────────────
# Do cascades at one device spread to other devices in the SAME FACILITY on the
# SAME DAY?  Measures within-facility co-cascade rate (contagion signal).
# FACILITY_ID and FACILITY_NAME are already in the Gold grain.

if "FACILITY_ID" not in df.columns:
    print("FACILITY_ID not in Gold — facility contagion analysis skipped")
    FACILITY_RUN = None
else:
    with mlflow.start_run(run_name="facility_cascade_contagion") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "facility_cascade_contagion"})

        # ── 1. Facilities with most cascade days ──────────────────────────────
        fac_counts = (df.groupby(["FACILITY_ID","transit_day"])
                       .size().reset_index(name="n_devices_cascading"))
        fac_total  = (fac_counts.groupby("FACILITY_ID")
                      ["n_devices_cascading"].agg(
                          cascade_days  = "count",
                          total_devices = "sum",
                          avg_devices   = "mean",
                          max_devices   = "max",
                      ).sort_values("cascade_days", ascending=False))
        if "FACILITY_NAME" in df.columns:
            fn = (df.drop_duplicates("FACILITY_ID")
                    .set_index("FACILITY_ID")["FACILITY_NAME"])
            fac_total["facility_name"] = fac_total.index.map(fn)
        fac_total.to_csv("/tmp/ps2_facility_cascade.csv")
        mlflow.log_artifact("/tmp/ps2_facility_cascade.csv")

        print("\n── Top 20 Most Cascade-Affected Facilities ─────────────────────────────")
        print(fac_total.head(20).to_string())

        # ── 2. Contagion signal: days where 2+ devices cascade together ────────
        multi_dev  = fac_counts[fac_counts["n_devices_cascading"] > 1]
        single_dev = fac_counts[fac_counts["n_devices_cascading"] == 1]
        multi_pct  = 100 * len(multi_dev) / max(len(fac_counts), 1)
        print(f"\n── Facility-Day Contagion Signal ────────────────────────────────────────")
        print(f"  Total facility-cascade-days     : {len(fac_counts):,}")
        print(f"  Single-device cascade days      : {len(single_dev):,}  ({100-multi_pct:.1f}%)")
        print(f"  Multi-device (contagion) days   : {len(multi_dev):,}  ({multi_pct:.1f}%)")
        print(f"  Max devices cascading same day  : {fac_counts['n_devices_cascading'].max()}")

        # ── 3. Contagion distribution + facility cascade histogram ────────────
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        (fac_counts["n_devices_cascading"].value_counts()
         .sort_index().head(20).plot(kind="bar", ax=axes[0],
                                     color="#E65100", edgecolor="white"))
        axes[0].set_title("Devices co-cascading per facility-day\n(1 = isolated, 2+ = contagion)")
        axes[0].set_xlabel("# devices cascading same day in same facility")
        axes[0].set_ylabel("Facility-day count")

        clip95 = fac_total["cascade_days"].quantile(0.95)
        fac_total["cascade_days"].clip(upper=clip95).plot(
            kind="hist", bins=30, ax=axes[1], color="#1565C0", edgecolor="white")
        axes[1].set_title("Cascade days per facility (clipped @95th pct)")
        axes[1].set_xlabel("Total cascade days")
        plt.suptitle("Facility-Level Cascade Contagion Analysis", fontsize=11)
        plt.tight_layout()
        save_and_log(fig, "/tmp/ps2_facility_contagion.png", run)

        # ── 4. Top 20 highest-contagion facility-days ─────────────────────────
        top_contagion = (fac_counts.sort_values("n_devices_cascading", ascending=False)
                         .head(20).copy())
        if "FACILITY_NAME" in df.columns:
            top_contagion["facility_name"] = top_contagion["FACILITY_ID"].map(fn)
        print(f"\n── Top 20 Highest-Contagion Facility-Days ───────────────────────────────")
        print(top_contagion.to_string(index=False))

        # ── 5. Monthly contagion rate trend ───────────────────────────────────
        fc_ts = fac_counts.copy()
        fc_ts["transit_day"] = pd.to_datetime(fc_ts["transit_day"])
        monthly_multi = (fc_ts[fc_ts["n_devices_cascading"] > 1]
                         .set_index("transit_day").resample("ME").size()
                         .rename("multi"))
        monthly_all   = fc_ts.set_index("transit_day").resample("ME").size().rename("all")
        monthly_rate  = (monthly_multi / monthly_all).fillna(0).rename("contagion_rate")

        fig, ax = plt.subplots(figsize=(14, 5))
        monthly_rate.plot(ax=ax, color="#E53935", linewidth=2, marker="o", markersize=4)
        ax.set_title("Monthly Facility Contagion Rate\n"
                     "(% facility-cascade-days where 2+ devices cascade together)")
        ax.set_ylabel("Contagion rate")
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda y, _: f"{y:.0%}"))
        ax.axhline(monthly_rate.mean(), color="gray", linestyle="--",
                   label=f"Mean: {monthly_rate.mean():.1%}")
        ax.legend(); plt.tight_layout()
        save_and_log(fig, "/tmp/ps2_facility_contagion_trend.png", run)

        print(f"\n  Monthly contagion rate (2+ devices same facility-day):")
        for dt, rate in monthly_rate.items():
            bar = "█" * int(rate * 30)
            print(f"    {dt.strftime('%Y-%m')}: {rate:>5.1%}  {bar}")

        # ── 6. Top facility deep-dive ─────────────────────────────────────────
        top_fac      = fac_total.index[0]
        top_fac_name = fac_total.iloc[0].get("facility_name", str(top_fac))
        df_top_fac   = df[df["FACILITY_ID"] == top_fac]
        print(f"\n── Top Facility: {top_fac_name} ({top_fac}) ────────────────────────────")
        print(f"  Cascade days     : {fac_total.loc[top_fac,'cascade_days']:,}")
        print(f"  Avg devices/day  : {fac_total.loc[top_fac,'avg_devices']:.2f}")
        print(f"  Max devices/day  : {fac_total.loc[top_fac,'max_devices']}")
        print(f"  Unique devices   : {df_top_fac['DEVICE_ID'].nunique()}")
        print(f"  Cascade days per device:")
        print(df_top_fac.groupby("DEVICE_ID").size()
                        .sort_values(ascending=False).to_string())

        for metric, val in [
            ("n_facilities_affected",     int(fac_total.shape[0])),
            ("pct_multi_device_fac_days", round(multi_pct, 2)),
            ("top_fac_cascade_days",      int(fac_total.iloc[0]["cascade_days"])),
            ("mean_monthly_contagion_rate",round(float(monthly_rate.mean()), 4)),
        ]:
            mlflow.log_metric(metric, val)

        FACILITY_RUN = run.info.run_id
    print(f"\nFacility contagion analysis logged (run={FACILITY_RUN})")

In [ ]:
# ── CELL 21 : Summary ─────────────────────────────────────────────────────────
print("=" * 70)
print("PS2 — Cascade Chain Pattern Analysis — Complete")
print("=" * 70)
print(f"Gold table       : s3://{GOLD_BUCKET}/{GOLD_PREFIX}")
print(f"                   (cascade days only — chain_length >= 2)")
print(f"Analysis grain   : (device_id, transit_day) cascade days")
print(f"Architecture     : CUBIC-MARS-AWS-v6  |  Analysis-only (no endpoint)")
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(f"Feature Store    : {FS_GROUP_NAME}")
print()
print("── Analysis modules completed ───────────────────────────────────────────")
print(f"  CELL 10  Cascade Propagation — Time-Window Analysis")
print(f"           Buckets : {list(TIME_WINDOWS.keys())}")
print(f"           Outputs : chain length/velocity by window, top-20 devices,")
print(f"                     device×window heatmap, top device monthly trend")
print()
print(f"  CELL 11  Subsystem Phi (φ) Correlation Matrix")
print(f"           Phi = Matthews correlation for binary subsystem co-occurrence")
print(f"           Outputs : top-10 pairs, bottom-5 neg pairs, per-device phi")
print(f"                     for top-10 cascade-active devices + heatmaps")
print()
print(f"  CELL 12  Conditional Probability P(B | A, T)")
print(f"           P(subsystem B fails | A fails, time-window T)")
print(f"           Outputs : 2×3 heatmap grid, top-10 pairs, per-device P(B|A)")
print()
print(f"  CELL 13  Markov Chain Transition Probabilities")
print(f"           P(next_subsystem | current_subsystem) overall + per category")
print(f"           Top-20 devices, per-device Markov for top-5 devices")
print(f"           MLflow run : {MARKOV_RUN}")
print()
print(f"  CELL 14  HMM — 3-state Gaussian cascade severity regimes")
print(f"           States : Minor / Moderate / Critical cascade")
print(f"           Outputs : emission distributions, transition heatmap,")
print(f"                     dwell times, state prevalence")
print(f"           MLflow run : {HMM_RUN}")
print()
print(f"  CELL 15  Association Rules — co-failing subsystem pairs")
print(f"           Apriori adaptive support, overall + per category")
print(f"           Outputs : leverage, conviction, bubble chart, lift bar chart")
print(f"           MLflow run : {RULES_RUN}")
print()
print(f"  CELL 16  Network Graph — cascade relationship visualization")
print(f"           Directed NetworkX, betweenness centrality (YlOrRd),")
print(f"           Kamada-Kawai layout, top-10 3-step paths, hub roles")
print(f"           MLflow run : {NETWORK_RUN}")
print()
print(f"  CELL 17  Cascade Ignition & Termination Analysis")
print(f"           first_subsystem (where to intervene) vs last_subsystem (outcome)")
print(f"           Outputs : ignition/terminus bar charts, first→last cross-heatmap,")
print(f"                     self-cascades, worst device per ignitor")
print(f"           MLflow run : {IGNITION_RUN}")
print()
print(f"  CELL 18  Device Recurrence — Chronic Cascade Offender Profiling")
print(f"           cascade_rate = cascade_days / active_days per device")
print(f"           Outputs : top-20 chronic offenders, rate distribution,")
print(f"                     monthly trend for top-5 devices, chronic classification")
print(f"           MLflow run : {RECURRENCE_RUN}")
print()
print(f"  CELL 19  Business Impact Scoring")
print(f"           score = chain_length × type_weight (cash=3, gate=2, printer=2)")
print(f"           Outputs : top-20 by total/avg impact, impact by ignitor,")
print(f"                     time-window breakdown, high-impact cascade composition")
print(f"           MLflow run : {IMPACT_RUN}")
print()
print(f"  CELL 20  Facility-Level Cascade Contagion")
print(f"           Multi-device co-cascade rate within same facility-day")
print(f"           Outputs : top-20 affected facilities, contagion distribution,")
print(f"                     monthly contagion rate trend, top facility deep-dive")
print(f"           MLflow run : {FACILITY_RUN}")
print()
print("── MLflow artifacts logged ──────────────────────────────────────────────")
print("  ps2_eda.png / ps2_window_chain_length.png / ps2_cascade_velocity.png")
print("  ps2_window_summary.csv              — time-window stats")
print("  ps2_top20_devices.csv               — top-20 cascade-active devices")
print("  ps2_top10_device_window_heatmap.png — device × time-window heatmap")
print("  ps2_phi_overall.png/csv / ps2_phi_pairs.csv")
print("  ps2_phi_topdevice.png/csv           — phi for top device")
print("  ps2_per_device_phi_summary.csv      — phi top-5 pairs per device")
print("  ps2_cond_prob_grid.png              — 2×3 P(B|A,T) heatmap")
print("  ps2_cond_prob_topdevice.png/csv     — P(B|A) for top device")
print("  markov_transition_all.png/csv / markov_transition_by_cat.png")
print("  markov_transition_topdevice.png/csv — Markov for #1 device")
print("  ps2_top20_devices_markov.csv        — top-20 devices (Markov split)")
print("  top_cascade_paths.csv               — top-20 cascade paths")
print("  hmm_transmat.csv / ps2_hmm.joblib")
print("  ps2_hmm_emission_dists.png / ps2_hmm_transmat_heatmap.png")
print("  association_rules_all.csv / ps2_arules_bubble.png")
print("  ps2_network_all.png / ps2_network_centrality.csv")
print("  ps2_ignition_termination.png / ps2_first_last_heatmap.png")
print("  ps2_first_last_cross.csv")
print("  ps2_device_recurrence.csv / ps2_device_recurrence_dist.png")
print("  ps2_top5_device_monthly.png")
print("  ps2_device_impact.csv / ps2_subsystem_impact.csv")
print("  ps2_impact_distribution.png / ps2_top5_device_impact_timeline.png")
print("  ps2_facility_cascade.csv / ps2_facility_contagion.png")
print("  ps2_facility_contagion_trend.png")
print()
print("── Feasibility note ─────────────────────────────────────────────────────")
print("  15% gap: SVN_STAGE.CMDB_CI / CMDB_REL = 0 rows")
print("  Impact : shared_infra_device_count + dependency_depth unavailable")
print("  Status : analysis proceeds on chain-derived features only (sufficient)")
print("=" * 70)


In [ ]:
# ── FINAL CELL : Export notebook with all outputs to HTML ───────────────────
# Permanent fix for SageMaker Studio output stripping.
# Run this once at the end of a full notebook run.
# The HTML file is saved to EFS (/home/sagemaker-user/PS2/) and persists
# across sessions, instance stops, and kernel restarts.
# Open it in any browser to see all cell outputs, tables, and charts.

import subprocess, os, datetime

_nb_path  = f"/home/sagemaker-user/PS2/PS2.ipynb"
_ts       = datetime.datetime.now().strftime("%Y%m%d_%H%M")
_out_html = f"/home/sagemaker-user/PS2/PS2_output_{_ts}.html"

if os.path.exists(_nb_path):
    _result = subprocess.run(
        ["jupyter", "nbconvert", "--to", "html", _nb_path, "--output", _out_html],
        capture_output=True, text=True
    )
    if _result.returncode == 0:
        _size = os.path.getsize(_out_html) / 1024
        print(f"[HTML EXPORT] Saved \u2192 {_out_html}  ({_size:.0f} KB)")
        print(f"  Open in browser \u2014 all cell outputs preserved permanently.")
    else:
        print(f"[HTML EXPORT] nbconvert error:")
        print(_result.stderr[-500:] if _result.stderr else "(no stderr)")
        _log = f"/home/sagemaker-user/PS2/PS2_run_{_ts}.log"
        print(f"  Fallback: write key results to {_log} manually.")
else:
    print(f"[HTML EXPORT] Notebook not found at {_nb_path}")
    print(f"  Actual path on EFS may differ \u2014 update _nb_path above.")
    print(f"  Run: !find /home/sagemaker-user -name 'PS2.ipynb' to locate it.")
